# Convert MHA weights to grouped-query attention

Grouped-query attention (GQA) keeps $H$ query heads but only $H_{kv}$ key/value heads: query heads come in groups of $r = H / H_{kv}$ consecutive heads, and each group shares one KV head. A common way to get a GQA model is to **convert** a multi-head checkpoint: average the key (and value) projections of the heads in each group, then fine-tune briefly. Implement the conversion and the GQA forward pass.

Write `gqa_from_mha(x, Wq, Wk, Wv, Wo, n_heads, n_kv_heads)` in PyTorch.

## Inputs (float64; projections are $XW$, no biases)
- `x`: `(B, T, D)`. $H$ = `n_heads` divides $D$, $d_h = D / H$. $H_{kv}$ = `n_kv_heads` divides $H$.
- `Wq`, `Wk`, `Wv`, `Wo`: `(D, D)`, the **MHA** weights. Head $h$ of `Wk` is columns $h d_h : (h+1) d_h$ (same for `Wq`, `Wv`).

## Conversion
KV head $g$ of the new `Wk_g` (shape `(D, H_kv d_h)`, columns $g d_h : (g+1) d_h$) is the **mean** of MHA key heads $g r, \dots, g r + r - 1$. Same for `Wv_g`. `Wq` and `Wo` are unchanged.

## Forward pass
1. $Q = xW_q$ split into $H$ heads; $K = xW_{k,g}$, $V = xW_{v,g}$ split into $H_{kv}$ heads.
2. Query head $h$ uses KV head $\lfloor h / r \rfloor$. Scores $Q_h K_g^\top / \sqrt{d_h}$, **causal** (position $i$ sees $j \le i$), softmax over keys, times $V_g$.
3. Concatenate the $H$ head outputs in order to `(B, T, D)` and multiply by $W_o$.

Share each KV head across its group by reshaping and broadcasting, not with `repeat_interleave`.

## Output
A tuple `(y, Wk_g, Wv_g)`: `y` is `(B, T, D)`; `Wk_g`, `Wv_g` are `(D, H_kv d_h)`.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def gqa_from_mha(x, Wq, Wk, Wv, Wo, n_heads, n_kv_heads):
    """Return (y (B,T,D), Wk_g (D, H_kv*dh), Wv_g (D, H_kv*dh))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0, 0.0], [0.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 3.0], [0.0, 2.0]], "dtype": "float64"}, {"__nd__": [[2.0, 0.0], [0.0, 4.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, 2, 1], "e": {"__tup__": [{"__nd__": [[[1.0, 1.0], [1.5, 1.2689414213699952]]], "dtype": "float64"}, {"__nd__": [[2.0], [1.0]], "dtype": "float64"}, {"__nd__": [[1.0], [2.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[2.0, 0.0, 0.0, 0.0], [0.0, 2.0, 0.0, 0.0], [0.0, 0.0, 2.0, 0.0], [0.0, 0.0, 0.0, 2.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]], "dtype": "float64"}, 2, 2], "e": {"__tup__": [{"__nd__": [[[2.0, 0.0, 2.0, 0.0], [0.6604769013466861, 1.3395230986533138, 0.6604769013466861, 1.3395230986533138]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[2.0, 0.0, 0.0, 0.0], [0.0, 2.0, 0.0, 0.0], [0.0, 0.0, 2.0, 0.0], [0.0, 0.0, 0.0, 2.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-1.2316649689147774, 0.2671189477208684, -0.006926123563646536, 0.5015352951547885, -1.3267282928093849, 1.107768166657506, 0.0937548811679036, -1.1708180763562224], [-1.3582403949601034, -1.3065640246645707, -0.7176152818724516, 1.1856213579293313, 0.8945337459684365, -0.5440478173910653, -0.5465879239818061, 1.9295877402585404], [0.9027885949894684, -0.8033905796046447, -0.14971196448317786, -0.43029158025002134, 0.15154145753356082, 0.8791498586600328, -0.27489147947244597, 0.36928102698558257], [0.5682654605670364, 0.7835076560925684, -0.6304340955306936, -0.221796455744778, 1.9723534034631014, -0.18596806121184997, 0.029913094799628087, 0.44094825078674765], [-0.5772605523613432, -0.3897476006393902, 1.6782279840043126, 1.2964559909835798, -1.1034374089422054, 0.0651657951870844, -1.2894297037907945, -1.5317475242426173]], [[0.0027364725013915097, 1.524554203577688, -0.08182613481761292, 1.4702009910257487, -1.0568504078749414, 0.13626829105665683, 0.08094594493870162, 0.08619597103012647], [0.9934001102540754, 0.5733787615494427, -0.5490205679347314, -1.3512770677788306, -0.5538102761652128, -0.020585458484185266, -0.5506707019933543, -1.5250804770169624], [-1.763631886148366, 0.8112818236152496, 0.25643114886432494, -0.5855581432194162, 0.4408157005034306, 0.49441833246945965, -0.21350892697386348, -0.42714517140046665], [1.1488160535822982, 0.8532398591206586, 1.4325295940128009, 1.0122079867940328, 0.2584077777588144, -0.44913400646498, -0.34264115212031465, -0.5945685496558906], [0.06525478330412512, 0.2091742914513544, 1.1665877402852103, 1.3475943515544997, -0.0180094929260246, 0.38995880499595187, -1.1755001136755354, -0.6776876568181975]]], "dtype": "float64"}, {"__nd__": [[-0.4345027555788941, 0.05068158902603183, -0.2152427654797988, -0.30963958373788264, -0.01168962703379484, -0.7499511307034995, -0.03912301894976564, 0.18131152752221247], [0.49303030927067354, -0.4711677350224486, -0.17454176436007413, -0.8236990171749352, 0.5272331724265682, -0.08472546469856224, 0.0854261446707892, -0.6158062139771587], [-0.1094929699406684, -0.2544157134497927, -0.49712470041996326, -0.4835728192369726, -0.39487934204557346, -0.36334560180692743, -0.7195091404892864, 0.5091088610712567], [-0.031725591089916905, 0.024090368681454092, 0.017827036908486128, -0.13187132350784822, -0.07463289869644277, 0.17057803455607543, -0.15745735288555984, 0.4556196426414465], [-0.13760474868603034, 0.40826598187160723, -0.4142305057905882, 0.06203163246653577, 0.22203632996161737, 0.4722430125970803, 0.4828987802044815, -0.4358275840741782], [-0.3070218790513073, 0.5691273913702019, -0.19743721454755514, -0.6485195412799228, 0.3913940287600444, 0.1888880233925685, -0.5232623771313513, -0.2809451970793628], [0.4571879653673011, -0.22454297026717912, -0.26739585525968723, -0.2106187328253192, 0.4656376910599986, -0.7034225693122567, 0.07776213874346223, -0.030499205269061892], [0.10368143054149562, 0.20530293779761172, -0.0688517297799721, 0.147057491203028, 0.09193124739333981, 0.47565250543348303, -0.9618241951049469, -0.12539441922144787]], "dtype": "float64"}, {"__nd__": [[-0.02888427887526057, -0.5839818134034956, 0.4867141773520067, 0.1573154775220496, -0.7036078005704854, -0.14353199453957838, 0.23842740593358452, 0.01030932921549944], [-0.31727109766597245, 0.21002383708745598, 0.10003737622232789, -0.3778025447402815, -0.10637656816362447, 0.4910361602345228, 0.4477872171414469, -0.38524623389963597], [-0.25777457665053083, 0.5372569429144255, 0.220976744507846, 0.011730864279385486, -0.12847651984086392, 0.07099664447248542, 0.3546424907343125, -0.9152803484713566], [0.3318812594985054, -0.35983614537260983, -0.5140166485555239, -0.2603504853696354, -0.4286599335932973, -0.3964654144079791, 0.015555666176362476, -0.16950008603467223], [0.38319377965876783, -0.6030728421249825, 0.3087238376761778, -0.4713673718322906, 0.0638503457494293, -0.12926001858387667, 0.36902810817305776, -0.49367083391107613], [0.03318380734252874, 0.6143552289150775, 0.5042239498528168, -0.09756566285643997, 0.3082845516731102, 0.4349120404862878, 0.21404455479156695, 0.4851847447855307], [0.40740482826219654, -0.8647439405152841, 0.2709483162077537, 1.4238913492120586, -0.12430907842694641, -0.09741449167187867, 0.04175280707338323, -0.2907007022716224], [-0.1058296921059449, -0.017438463234326577, 0.8125922973824444, -0.0733294304525988, 0.24669016888817166, -0.055623769522512446, -0.34658687174210184, -0.24029544368267014]], "dtype": "float64"}, {"__nd__": [[-0.42240267700839046, 0.5621006464505848, -0.07665010982268128, -0.55538277771238, 0.01255403019972087, 0.833182372434632, 0.12314683742040611, 0.0034693694895088383], [0.13038121205776407, -0.1373283011155618, -0.39290235111314525, 0.5356748117768521, -1.3018823150672474, -0.19568625381051855, -0.342273398781796, -0.07020552664356948], [-0.2806215854073428, 0.46296396731335376, 0.63585295167569, 0.07370002240200979, -0.4901598167320601, -0.28063418902766263, -0.13079045690789756, -0.3458772365837855], [-0.483079077964241, 0.7050909278970701, 0.3846929679968453, -0.37650381581832437, -0.34063889721154017, 0.03410004311793817, -0.25712358833843224, 0.23722465087913325], [0.32319309907929816, 0.025188759199270185, -0.3600910919785898, -0.11683936605256702, -0.6458300786265216, -0.13912364333128732, 0.7239328588449901, -0.21545345833168825], [0.2754746457577881, 0.48520627642231845, 0.6458076781240512, -0.524804520562211, -0.07699449510113382, -0.21973071457718563, 0.5465461219402261, 0.43837036591186485], [-0.41754034560533615, -0.21688596454949438, 0.1464289615345806, 0.823590503295948, -0.1422015158698805, 0.4916885281308531, -0.3304163372858117, -0.4852387585910134], [0.33012128668947177, -0.3076987375088893, -0.6202608508217043, 0.5015178820952786, 0.24470731875016638, -0.5098370054498601, 0.3431857113914466, 0.17348097810736582]], "dtype": "float64"}, {"__nd__": [[0.22355526289645256, -0.4241628381607876, -0.1657192028343395, -0.851103882264662, -0.851711950594898, 0.32150986384754593, -0.18750012573783223, 0.08208326194752465], [-0.09085354816589025, -0.21227040557474297, -0.2065957387416508, 0.19432014811438408, -0.09849787192579479, -0.6176028416432047, 0.44646213004801477, 0.5540758918532391], [0.3045285114173206, 0.5906977930256007, 0.07328895320529778, 0.008999978977140463, 0.45140220830856226, 0.21501529269636954, -0.4751344279489176, -0.07020360672961949], [0.035255626425160815, 0.8738480078306123, 0.01782182291510138, -0.3073730121385047, -0.4333767130447581, -0.06608033924277235, -0.2591469598018031, -0.050024980279459424], [-0.40564676542475797, -0.6494306364437721, -0.11728586421696521, 0.13304621643890013, 0.04164952198205213, -0.6147037744378376, -0.1272567639180594, 0.30959521218464986], [0.10763724138679186, -0.26275020077274863, -0.1910887034705646, -0.19078093672811317, 0.07531780867368759, -0.1573342576044676, 0.17115618871986302, 0.058963027552394], [0.05213041589491022, 0.3043247865444039, 0.32736092490259927, -0.1670009386951008, -0.12412237528861202, 0.35842569176298644, -0.1276883435430203, -0.500066880726409], [-0.23892129197361092, -0.4309657483619349, -0.021366634976476773, 0.03186086707170697, 0.5228287194669459, 0.1606084907685191, 0.2325365114384068, 0.5837255614693428]], "dtype": "float64"}, 4, 2], "e": {"__tup__": [{"__nd__": [[[0.6965405844636045, 0.37098809689516116, -0.2459096572178135, -0.7904765145955445, -0.32610628860720786, 0.6026545282666819, -0.4213860058800652, 0.2161026640738477], [0.769445362135126, 0.8205529222808683, 0.12268994523951574, -0.8660896986730364, -0.5242396622075446, 0.8753894264974098, -0.6080087368176746, -0.5012693600520863], [0.04866879297689169, -0.47008839594489826, 0.06035305546982474, -0.6752059326418337, -0.5297633059692906, 0.17835973474553582, -0.4922376072305368, -0.12108802262642888], [0.17741691084658695, 0.059285102481512725, 0.14046203159221085, -0.3723969451925866, -0.2889587416572466, 0.29937344843558, -0.3408207826741946, -0.3954968140058954], [-0.1475550754688208, -0.5593956044765664, -0.08418591805765506, -0.13822082078650388, -0.11188086844342854, -0.08734796649436931, -0.09005504676083889, 0.2119940190074927]], [[0.4083434686140689, 0.7897049084331064, -0.48828385810228214, 0.14776848609380183, 0.03578491044268994, -0.10556376164795744, 0.7756145330782388, 0.7547650905521764], [0.3171866924457477, 0.2725208553324139, -0.22792714863244362, 0.05328229254109686, 0.34880981210426926, 0.3492850999461826, 0.5458840266527498, 0.3756782112511111], [0.3052218332946503, 0.19086916090073516, -0.26793107062501453, -0.06498837153955274, -0.10177630148176119, 0.041838318074495076, 0.07163389158217344, 0.2093107120683902], [0.4543528477509369, 0.7480244556790641, -0.14061852809322686, -0.07420177397746738, -0.039883104927891655, 0.4112947222738439, 0.3799244449294617, 0.14151482122870152], [0.3827285482047412, 0.44758929743586523, -0.2033547690418458, -0.1752801869914574, -0.026510809189111875, 0.30053806571373504, 0.2141314636506191, 0.2615876710027768]]], "dtype": "float64"}, {"__nd__": [[0.22891494923837308, -0.21333316794072302, -0.23259019731845046, -0.06661133266203947], [-0.10861686072182228, -0.08388935382641277, 0.17070532448891124, 0.05289496316744341], [-0.01839891607134242, 0.2744939035969055, 0.11308298544672428, -0.4221418519994356], [-0.09106769452850924, -0.3100933153711226, -0.20655213370846742, -0.2829827502213257], [0.3459588086674728, -0.5372201069786365, 0.2164392269612435, -0.3114654262474764], [0.26870387859767275, 0.2583947830293188, 0.26116455323233856, 0.46004839263590924], [0.3391765722349751, 0.27957370434838724, -0.04127813567678159, -0.19405759697175054], [0.35338130263824974, -0.04538394684346269, -0.04994835142696509, -0.1479596066025913]], "dtype": "float64"}, {"__nd__": [[-0.24952639341553587, 0.0033589343691023954, 0.06785043381006349, 0.4183258709620704], [-0.1312605695276906, 0.19917325533064512, -0.8220778569245217, -0.132945890227044], [0.1776156831341736, 0.2683319948576818, -0.31047513681997885, -0.31325571280572406], [-0.04919305498369783, 0.16429355603937285, -0.2988812427749862, 0.13566234699853572], [-0.018448996449645816, -0.045825303426648416, 0.03905139010923425, -0.17728855083148778], [0.46064116194091964, -0.0197991220699463, 0.23477581341954615, 0.10931982566733961], [-0.13555569203537776, 0.3033522693732268, -0.2363089265778461, 0.0032248847699198613], [-0.1450697820661163, 0.09690957229319463, 0.2939465150708065, -0.16817801367124713]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.7279235046961556, -0.8917006968881649, -0.09770470936575483, -0.6713134449611673, -0.21694063841781236, 1.1539034286423295, 2.816543042840967, 0.7882928220084852], [0.22549296270326338, -0.7381506468226016, 0.2493314706211564, 1.5276219240557176, -0.7919186069465554, 0.40657453189899667, 0.8057714095088087, 0.9353095999578672], [2.3535755144935004, 0.0011273984989907469, 0.2394411488955997, -1.0382267007688286, -0.47361071825355755, -0.9314520988783755, 0.08250977624408094, 0.4278387270525196], [0.6690530286344901, 0.7707390729353415, -0.511815061125778, -1.008535311566219, -1.009839016375777, -0.29844760499921164, -0.45233121867329373, -1.1882069485945783], [0.38512695463841773, -1.1048950137605775, -0.13688104293395137, -0.605169276103293, 0.3226447799314622, 0.4758031200511162, -0.9136973998041933, 0.4029004289210695], [1.9751986520161384, 0.4467924685771348, -1.0718232234924387, 2.1719549002925027, -0.9415614890484273, 0.7091088898899445, -1.0639488720971693, 0.7279537901456985]]], "dtype": "float64"}, {"__nd__": [[-0.06068514216081662, 0.20557370181950585, 0.08903046663049825, -0.5929901215847265, -0.6396505375282922, -0.3365002565040515, 0.3461589285246012, 0.12475607685424434], [0.47856943279209097, 0.4835180774086465, 0.6776375863248983, -0.37751308621957713, -0.4613949154802387, -0.44035252109632533, 0.5353706660522863, -0.1630660508652814], [0.1208474935753876, 0.7630170099108955, -0.15200992403863867, -0.2447339610598098, -0.37447942279376245, 0.3864272677142764, 0.41084021138582144, 0.040882815744703066], [-0.14532016679873805, -0.04330957053783521, 0.15417965186563715, 0.42651807848865797, -0.3910632640935646, -0.3028977706843624, 0.3555298215548506, 0.33589085591855794], [0.01845073579269953, -0.05406106591132287, 0.39731274215820855, 0.05493146975353613, -0.6801574025897934, -0.41953275336394535, 0.2848486616875065, -0.0648234241441109], [0.16781886982372274, -0.6841399353406273, 0.45286501831073694, 0.0989618989771908, -0.0788044561252602, 0.03126316077206235, -0.12813996204242445, -0.09115245089237288], [-0.9052593675389948, -0.4081137217293397, 0.4278088680057708, -0.08727182146121153, -0.35677120948230323, 0.19194779604918297, 0.5912774236228165, -0.011145469009518169], [0.32748906056029825, -0.2563158563988162, -0.7143407364721105, 0.6191393644455502, -0.29406329000418346, 0.3197729843120894, 0.41651338183173303, 0.66701777833719]], "dtype": "float64"}, {"__nd__": [[-0.20837676216257417, 0.4424440436947813, -0.37003493395512743, -0.5694040995003699, -0.6400695179290306, 0.0908176396762409, -0.5815802580067367, 1.0646029017906964], [0.4382117419262951, 0.4288167139223267, 0.15290064239679554, 0.3789859418440573, -0.16547728694663988, -0.4103892777050906, -0.3232695705734765, 0.25128166853116185], [0.0419823332274748, 0.1684780786060789, -0.5275144978909919, 0.3349719541725508, 0.5143336872778514, 0.16545139722964405, -0.9011339860380639, -0.2139099138628019], [-0.000898669202860891, 0.4399934256061019, 0.08540101316010926, 0.32624294517851515, -0.3785791918743846, -0.26886452415571255, -0.26890730293209175, 0.5510315938564574], [-0.6978659495029221, -0.1610778169377236, -0.7282062018899561, 0.6589347939436676, 0.30071665465714426, -0.3180852817357991, 0.2933051548322334, 0.2896268363216759], [-0.09331979146340783, 0.1333296067150092, -0.07929854384235212, -0.07777179298166914, -0.6892611994223738, -0.3738135523950791, 0.28264269821534965, 0.06448987387807907], [0.8688759074032119, 0.5195995639792702, 0.1811748246130541, -0.7897055753074597, -0.26453430574857983, 0.07006252274280099, 0.7755977122546679, 0.1792630026565203], [-0.5967076162284859, 0.3436800785997119, 0.41693321860461774, 0.5029918701070092, 0.586912441450296, -0.35267899763896443, -0.012609808428259692, 0.20160349841266356]], "dtype": "float64"}, {"__nd__": [[0.3653447814492934, -0.24246604046935213, 0.3996542503612419, 0.34527884120671604, -0.9070111042743951, 0.2641147015243685, 0.11167432878943558, -0.24160052928413048], [-0.17257072577606664, 0.19534067298154703, 0.6834944399131148, 0.2184606954198278, -0.6172107835169643, -0.17196060995236037, 0.019016629313505375, -0.987063779926288], [0.003099846901936227, -0.28382801125861773, -0.19321543039846198, -0.49132632769764756, 0.1299919048091913, -0.44997578864980153, -0.4658578191235385, 0.01742105112228653], [0.3113880723805893, -0.2466828014488344, -0.16346048070650565, 0.2878636302109378, 0.16371921655402302, 0.016720356255491214, 0.42158055905402025, 0.31824947363674366], [-0.43410349915703145, -0.45638620601786545, 0.7020063866427139, -0.3356991947667236, -0.792406136708967, -0.1769642361917876, 0.010756158911879676, 0.31436846206165914], [0.7368461239669646, -0.04976408073025201, -0.8436396709188936, -0.2976831692357966, 0.20745834244061087, 0.5577318568091213, -0.12300688332346632, -0.37821674386078397], [-0.19842644853990313, -0.6064696739396331, -0.4781057479552675, 0.8986499842533731, 0.6419188789095893, -0.2686890559453362, -0.5165494457432122, 0.3171154308105929], [0.2607876541941993, 0.18559849371482137, -0.7905692852123399, -0.056716295344889825, -0.3604860057218165, 0.42592301128244386, -1.0816435523413246, 0.3056967567017965]], "dtype": "float64"}, {"__nd__": [[-0.27009630833935727, 0.35355000341228404, -0.2200476034181935, 0.46191697766975487, -0.14268343192704333, -0.2761588275224818, -0.22563725306736468, 0.5601309867575724], [-0.33806022947135095, 0.1096840679521492, 0.3313523890135345, -0.4514311203263603, 0.01734498937626043, -0.11421423631408233, 0.598806596584556, -0.25488804066225307], [0.1849595733763134, -0.47703192151167556, -0.597128045804158, 0.2279033735952014, 0.30417613214513506, 0.271062062007699, -1.1231804596455992, 0.6256986046576378], [0.16865525002667026, 0.14487262298859382, 0.891813894034471, 0.35458783467309735, 0.06645827234850374, -0.42507741760749024, -0.054951748442312355, 0.37067770450112136], [-0.8372989365485687, 0.13669475900311298, -0.2086253177161552, -0.28086056357164985, 0.4058631946943117, -0.06794604413145784, -0.6201157922797703, -0.3674072051394927], [0.18945663186524908, 0.26286636006126046, -0.7436212085576757, -0.36087973661014583, 0.11998805438021061, 0.5421757476843497, 0.23695486890962228, -0.2765327785927795], [0.2889091595617816, -0.3711300563836659, 0.10971579290335372, -0.05188618236775923, 0.4537171731590879, 0.5328182813115623, 0.6147919664923287, 0.09959710839308296], [-0.0034039431924358064, 0.1566732546712674, 0.9426930727629852, 0.07092890512653359, 0.14865343177108184, 0.028453232855445116, -0.4070175401934833, 0.364845060965018]], "dtype": "float64"}, 4, 1], "e": {"__tup__": [{"__nd__": [[[0.5486724264020603, 0.6767443398952773, 1.5648715948623286, -0.5174970166143074, -0.6750324103671672, -0.3742354089114946, 1.3593869211246599, -0.6691292863916527], [0.3002802599795652, 0.5342055989889046, 1.2120518246534944, -0.3992714847944944, -0.2671696829665059, -0.1997254028419147, 0.7434937989901765, -0.3125950993101288], [0.224882772661642, 0.4871411178395777, 1.0844703963423175, -0.29795510386928237, -0.36051682955954645, -0.2783265622210808, 0.7028304943730299, -0.3305326633323087], [0.26740952184524724, 0.16503776553090171, 0.47787538024597354, -0.2418968577395288, -0.08327220799716319, 0.046468236541461816, 0.6048959352306356, -0.18158464150296208], [0.18603146591363834, 0.19841672853429904, 0.4187321317800577, -0.2082841558523137, -0.1220266790292918, -0.01154427974257051, 0.44133724632600607, -0.16948239521529004], [0.010021775527593518, 0.2055883348808175, 0.7261287167469157, -0.19156614148969336, 0.06768685010467065, -0.07460043897854378, 0.1323273854647411, -0.055974472145379955]]], "dtype": "float64"}, {"__nd__": [[-0.45001536801336717, 0.2571151214153372], [0.02559138170074357, 0.1621737616481138], [-0.2180831158559324, 0.11374787903636799], [-0.14074603771230698, 0.2621008601213405], [-0.20801258547587514, 0.1173496328979552], [-0.14480920912819606, -0.06344146619591498], [0.3902785346305885, -0.005195121482217041], [0.09863205884954203, 0.17389911237010508]], "dtype": "float64"}, {"__nd__": [[-0.007584435918606063, 0.03133174324440049], [-0.0218176100166027, -0.1863057553693184], [-0.13149537445271825, -0.3019272691209451], [0.18330684182053172, 0.09403766466358457], [-0.12843677257785124, -0.16367029372867936], [-0.005585521958696112, -0.041983034254427815], [-0.13779069083219836, 0.08515167129474915], [-0.49297779727032043, 0.215125491588543]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.6972098407332815, 0.19502157202914716, -0.5165874327361333, -0.3456100149521401, -0.41852600390659545, -1.537616267791916, -0.4595207914135334, 1.117755533868975, 0.5642425680747702, -0.30938753192246154, 0.7988415216776881, -0.2378382323251877], [-0.5913988043605395, 0.07525925875803292, 0.36594414769988454, 1.8815771140959943, 0.13791990938277246, 0.07610979002951217, 0.6220738732279774, -1.3058116686332462, 1.3727638376880817, 1.8237011151621312, -0.11412079748982777, -0.5077771568252193], [0.2054343741383729, 0.3939309110848392, -0.6881392959873576, -1.5867570155810429, -0.05116407086834034, -0.6426338449292359, 1.4672176075700645, -0.1070465432532403, 0.03189164555272928, 0.14574299639253607, -1.0593477879528466, -0.8966315794940495], [1.9560936169217749, 1.3950798829260203, -1.1729733056431548, 0.3632503861603359, -0.2873919006206359, 0.807747526719992, 0.995971591545557, -0.4339339228537332, 1.008580124508772, -0.4977903249611092, 0.3932628387867448, 1.6609418755977494]], [[0.6387891838422631, 0.23541748923607406, 1.4902147291314285, -0.5810400336956794, 0.6496943009937738, -0.2109520428279951, -1.569683893691026, -0.0921789040652725, 0.42429887153385876, -2.0316697390727, -1.9722496543137487, -0.15196179722277], [0.003988503706460209, -0.14049509382633385, 1.6914027090205332, 3.114068664892388, -0.2731290529132164, 0.283615653068101, -0.6187318908038841, 1.1269626296081507, -2.018984649830774, 1.1500650996458028, -1.8375786670601668, 0.34696462588584137], [0.44830451383418013, -1.0315192578875179, 0.18614009674492596, 1.5143280186492685, -0.06298906483014408, 0.25813764775029097, -1.1969042948453765, -0.36851957060004653, -1.6031523838654913, 1.4479721245820698, -0.25879846570763765, 1.0410621366674926], [0.9751162219490279, -0.3735150479950024, 0.6021662306289551, 1.0871231625536184, 1.3625839438753924, -2.8768735675293695, 1.530824135128372, 1.4033501170894311, 0.2604444382431099, -0.13656560538626195, 0.16203653793202571, 0.8698691362411076]]], "dtype": "float64"}, {"__nd__": [[-0.7769786798165431, 0.7252626091784847, -0.4874552012110133, 0.946939595643959, 0.18349263801829907, 0.4565278977886784, 0.138820974882608, -0.17785001842305392, -0.5328475895311235, -0.2596878254938705, 0.38086605517437117, -0.1526170496111954], [-0.013475158053315103, -0.04649148970473124, -0.3274620788591722, -0.20065337623048252, 0.5938742168765563, 0.5929305445389254, 1.1477064265538244, -0.34989447196662027, -0.43354803823292126, 0.06851115217056368, -0.5828762599530392, 0.15075686453817722], [0.6857253576603135, 0.164681047557166, -0.12452574112750961, -0.7533465658826688, -0.0040650829783852645, -0.41599242865848346, 0.16222586063035088, 0.08615647724101828, -0.8822644780441351, -0.552858165839931, -0.39045437882035194, 0.1007064503104423], [0.3297260254343104, -0.514474191269599, -0.23500493157498215, 0.1821328741519095, -0.6556605941583797, -0.38106338938322887, -0.31261530085520545, -0.31123295019777814, 0.5480661609035489, 0.07853819908044568, -0.6525820455624668, 0.5500788037038628], [-0.09673530233122982, -0.07309061924484017, 0.5748251362932264, 0.009330785927239621, -0.3290285693220828, -0.3379444267455028, -0.042459195633705156, 0.33844274901393034, 0.3276482674989825, -0.11934635976098354, -0.4888162453597586, 0.32563829769048724], [-0.6338705857608177, -0.1046066203107956, 0.31638384734632086, -0.4143776933033756, -0.05730283367852867, -0.36192590106206746, -0.31032903390002725, -0.20044267348597963, 0.24398703988207507, -0.2760659733098812, -0.26767472642895623, 0.5806021853390808], [0.552541228696455, 0.41530027370164657, -0.04046573182012422, -0.005341046180052943, 0.14604179396318293, 0.013379960452131351, 0.03572483195666796, 0.4309783611693632, 0.8317508805596598, 0.1384169577327112, -0.08759983447561308, 0.14582826322727557], [-0.05623452457732121, -0.28468941623603977, 0.11377494709732, -0.1464077434198496, 0.49952560632077364, 0.15887665271529106, 0.11663555943310144, -0.21625865335726707, 0.437068374932579, 0.5261817324086883, 0.22255947965395773, -0.22105042504813263], [0.06982497257056346, -1.1715080312985526, 0.14045820923206262, 0.3376231560713884, -0.5444304852128306, 0.15347622247276216, -0.48717949903954205, -0.17033565143616747, -0.24708029773134102, 0.6201117937389719, 0.23610449535329536, -0.027877576521449572], [0.00991865240519934, 1.0153733901198214, 0.005802113782628076, 0.14962007197505175, -0.9833428412526102, 0.5062567930142692, 0.07408877201866772, -0.2627013370659875, -0.11925781956285081, 0.21898497746093032, 0.14111412991680977, -0.39350459071386856], [0.07422148842592725, -0.03604756281262854, -0.22542561929885566, 0.803558139527293, 0.2022052606098762, -0.547288736425033, 0.045998974742245194, 0.010307363892743256, 0.1845440047229675, -0.2568854216527419, -0.704515243319889, -0.403474110308392], [0.789180791856626, 1.0611814957719576, -0.08666888713190152, -0.997206669765925, -0.37357487823086855, 0.14789052110635478, 0.48850984578305856, 0.2873296620976184, 0.4498562667481448, 0.27298098552355493, 0.03498444817535004, -0.09076793133044424]], "dtype": "float64"}, {"__nd__": [[-0.36256311355843557, -0.7368816169085531, -0.3499368667017779, -0.1505729051575957, 0.31725810063655224, -0.7149986303069702, 0.49637843305216517, -0.6148384936804027, 0.5958264578803236, -0.3331618419614173, 0.1571895683159255, -0.0951640132541599], [-0.012023510723493891, -0.4014785461032528, 0.08493633400293042, -0.16707662214723445, 0.22803939232587053, -0.0023058052508666176, -0.12954538750709096, -0.6424370300949176, -0.00130944725624895, -0.9369538926009299, 0.009489538906102497, -0.03221054329651574], [0.3551672020319839, -0.6899903853277368, 0.3433827010884257, -0.07590040186768302, 0.2216188096631395, -0.14192452648575013, 0.025889912710477216, 0.5706005275224828, -0.09937216690692874, 0.22242069424523622, -0.1258769968830082, 0.044181119561202235], [0.4544500767725379, -0.1038497478428359, -0.023242727193545382, 0.46859671526304236, 0.1826277478910403, -0.1906320957691591, -0.028451415439480394, -0.4538600302747704, -0.24361585506064937, -0.5419667799510319, 0.049984449663702885, 0.1716640718882682], [-0.7444579287117209, 0.39841503139296147, 0.14533626848648173, -0.27139640968996653, -0.33110329009770434, -0.4060577376206597, -0.13430980098640552, -0.05779337230192905, 0.3355654002129609, 0.9309153561572213, 0.06842437289075823, 0.2515915370183288], [-0.5246226083253992, -0.056351537212605496, 0.27775572924023145, 0.18272453756083676, -0.555769779921455, 0.46176802146689255, 0.16435454551492645, -0.5770519079581569, 0.2889509719305168, -0.41555173144219604, -0.5703760891515149, -0.4874769292226079], [-0.09971293198231446, 0.5547149695730113, 0.231266625048191, -0.22331934224128847, 0.5536228895286717, 0.12133066924953391, 0.35743718052639406, -0.6139571662327159, 0.2013135497580663, -0.1166753763895076, -0.5531491907900823, 0.28500446706142013], [0.044167260008835446, 0.06788110698459777, -0.2026524802882812, -0.46347985603141617, 0.4627319672698552, -0.0221483617988133, 0.2856533936869824, 0.0009813819042102487, 0.09637729524472495, -0.27421965328983194, 0.10993558155222134, -0.03369010261943646], [-0.23639564777286354, -0.10658495083807983, 0.08931539079775373, 0.15781677901763225, -0.0191656911369484, -0.40941112068617563, 0.20282819467527632, -0.20611711150906237, -0.9358622835616356, -0.5423699208620387, 0.11891015853364602, -0.2325393180189328], [0.1546594312627517, 0.17639140990505087, 0.7004067523916717, 0.2910977109903045, -1.471450739193938, -0.3523563574753367, -0.9470667634805011, -0.08527221324140612, -0.10423887986390944, 0.30483549564385826, 0.2282114088126278, 0.37503875809738835], [-0.22954327310632366, -0.3260717373835061, 0.0852864939954141, -0.18777581342925598, -0.26349670894503857, 0.4752158117294215, -0.1478546765976917, -0.16787988387633124, 0.14226818440887753, -0.1145006893042035, 0.36706823815786344, 0.45852139312242807], [-0.31041759653293033, 0.25362479184499187, 0.30115586242787523, 0.1718088089440879, -0.2661571693183952, 1.004586815482497, 0.12528246013274744, 0.23926258526049923, -0.28332813563833587, 0.7876654413244191, -0.21022471672321952, -0.43023654498822234]], "dtype": "float64"}, {"__nd__": [[-0.4741608444690541, -0.11548673894947346, -0.6119154439080443, -0.13700699508124853, -0.027118392552630585, 0.3696727487225553, 0.4101919305619949, 0.38077385022872945, 0.4025943049343633, 0.8973042760766203, 0.08947652351620719, 0.5070996588325385], [0.3229857433866239, 0.17833782927294306, -0.669014247247166, -0.03147613561172305, 0.634853892465219, -0.4015755082888271, -0.2509126084015698, -0.024049186855046773, 0.17967143259617502, 0.07275575882451703, -0.06307870795320149, -0.6048186993023857], [0.32545801159001614, -0.17562731552306884, -0.2030758668076192, -0.24313672881794057, 0.1909632688547169, -1.0289054801078894, -0.6200606796936061, 0.4521653975888971, 0.18431024759279832, -0.4291729902606239, 0.722057737684703, -0.3336097335295802], [-0.015703649018845203, 0.28456719504649425, -0.3552027759608355, 0.2237529916740092, 0.17925652671992046, -0.3435177223106396, -0.5290689276993653, 0.5616455436645477, -0.0052458291265314915, 0.7694049745857421, 0.29892773129872474, 0.1481658947755192], [0.012521689036348564, -0.32481573675310305, 0.1008703656550185, -0.24162958486990138, 0.5785929514180086, 0.4761738111243046, -0.6310154605583324, -0.37283048603880997, -0.3774678804143614, 0.3156148601834247, 0.19135823013032494, 0.20739432785452006], [-0.10102139830602051, 0.2952591372552799, -0.6295403734787898, 0.29987371380636, 0.09104918943087807, 0.04952008727524962, -0.15535612998578452, -1.1534307911118762, 0.39534989847050755, -0.4341908535162329, 0.1455783492880219, -0.35655141904564036], [0.23512297716676614, -0.03881213719587002, 0.3076994421646, -0.09218677567780545, 0.21955114360730935, 0.13335568956264965, -0.91911255869433, 0.4411859817734232, 0.13164591472076803, -0.07199203260703135, -0.12289739287996902, 0.15992034578008577], [-0.45401831965838346, -0.15432661780145712, -0.09546129574527593, 0.751025208711765, -0.4138389814647348, -0.7087746411101702, -0.3512624667755464, -0.2732620245069276, -0.22244851355327966, -0.10068852155887809, 0.2142139203725344, 0.11970083092507178], [0.005087866093103233, 1.0102944366184392, 0.456381452401937, 0.3250667894196157, -0.006435200125688533, -0.32885236944009444, 0.13068234121712136, -0.7822978012286759, -0.2881922696353632, -0.2534593518801601, -0.49654987431449316, 0.7743601085879913], [0.504799838660109, -0.1466240374108243, -0.46667721725553335, 0.3167900931384876, 0.04113639263528742, -0.06627015212721475, 0.3090564603745358, -0.37486404109935134, 0.1141339934169319, 0.9357778511491603, 0.30290601713499876, 0.20662408543794514], [0.7967335724061331, 0.4811058800688887, 0.6988559034760408, -0.20039370820636793, -0.24754809777725725, 0.048797599434682606, -0.29165947840968487, -0.4536649341770336, 0.6720131844423249, -0.12081700840027099, -0.6246970814736863, 0.10935439604944819], [-0.009211557637569694, -0.03328870595872351, 0.03626277561476214, 0.15734710827390436, -0.42189953473871145, -0.1070371226607197, 0.06919598414575079, 0.37237445978024003, 0.02841357933240234, -0.4404242530488408, 0.2707978676723521, -0.4661529930935487]], "dtype": "float64"}, {"__nd__": [[0.009062979443340226, 0.19407176679260346, -0.095978290016563, 0.28136344765154236, -0.08753849399520526, -0.25626158636520713, -0.10608830475171134, 0.08778685317731454, 0.22204031869692234, -0.6686221933001012, 0.12797063043948395, -0.1722605944211944], [0.4073525847997185, -0.7830417090968806, 0.13598659077665917, -0.2708340502608539, -0.20927347496240112, -0.4705875168977648, 0.4798221069927273, -0.5905675658847743, 0.2762099237604395, -0.6281428825063634, 0.21101543143629187, -0.6096616747679414], [-0.11605907915081924, 0.6227802973468535, 0.27199878498524227, -0.7140804661695332, 0.00019751767373172828, -1.2234878340848165, 0.10582242977607825, -0.8835809910877561, 0.17168793498917945, -0.15237062295692907, 0.26600852009447723, -0.289495832288934], [0.33531890980194684, 0.10800939320418204, 0.6760577686922714, -0.41250892635697795, -0.16437545219919614, -0.39511600389589574, -0.15566743404299413, 0.34548495587389083, -0.06261646214305579, -0.24235640781468024, -0.1312883579991104, -0.11165123709014271], [0.004632503374529656, 0.620396440313549, -0.06871808687356322, 0.8319128603188942, 0.24930203043603993, 0.2151252178665598, -0.46991603631642376, -0.2865994161118674, -0.2113119767897612, 0.3603920931614292, -0.5409793456566621, -0.2663708636425085], [-0.40181287409732935, 0.4269637034217392, -0.659538207763276, 0.22110824881564511, 0.16694808324762223, 0.09633423920954348, 0.6343052779288427, 0.2259215929280489, -0.5160145531724678, 0.299517978208616, -0.19957002446184235, 0.27773069877575596], [-0.23881341917988538, -0.13260722566578495, -0.6091018926742793, -0.17493860939636507, -0.5926461060915363, 0.05226792606124601, 0.08141263283583974, 0.7434459562153809, -0.44247790446471486, 0.2494755917726027, -0.008913104867866912, -0.32685234098548005], [0.8634829158117363, -0.6729097858025722, -0.23499218608524106, 0.3557519693401594, 0.08090805879408076, 0.09692561385636123, 0.2926913956317918, 0.4408939118924859, -0.8279993804808528, 0.1398546917084217, 0.008628428573845832, 0.33244884892405985], [0.657908772649312, 0.13728640250100957, -0.0377502256685923, 0.22197760311779363, -0.5976842352435379, 0.3837680696685546, 0.22379419790210653, -0.37453093078756644, -0.27912709486971227, -0.1198356447579124, 0.30142482367134593, 0.9078200066119266], [1.0926699786082796, -0.3603184151274952, -0.22936531130567275, -0.14820960531733302, 0.47004090287039013, 0.049808919441448496, -0.6796139637640761, 0.0691161236429441, -0.6019297090615917, 0.3259795183870857, -0.0037477025697887447, 0.47422014927138867], [0.1807000991806492, -0.3023782273060533, -0.4633465778615346, -0.5576061134229543, -0.12019888054583244, 0.7899000581778302, 0.7039033523886413, -0.38028219261088875, -0.45041273516664176, 0.7129746516889464, -0.32996566313839293, 0.032319208664865616], [0.03233405490571697, -0.20651674807477965, 0.4047200994367085, 0.3533917097446694, 0.13453938552485634, -0.6056342489406741, -0.5812247068135913, 0.13069715999578174, -0.21983662185322578, 0.3132404191484669, -0.21049678730802796, -0.4131902439029486]], "dtype": "float64"}, 6, 3], "e": {"__tup__": [{"__nd__": [[[0.802385909168632, -0.4154365030829086, 0.9990019481858381, -0.015866192062667428, 1.027406386089186, -2.5305509441198044, -1.8465334225798065, 0.3459416620908626, -0.029371638230187846, -0.5508264500096378, 0.03233418831785531, -1.1275565670544447], [1.6515089186351881, -0.6651582632511818, 1.5851322924831217, -0.6681347824720103, 0.5520484230269206, -2.9628271580172765, -2.542884110916594, -0.45717270964202855, -0.18417675786791357, -0.24844697570530144, -0.23584408919443886, -1.8174687955063347], [1.1623502605367806, -0.19131931092010868, 0.5802928407144896, 0.15514860456527607, 0.8621481938556541, -2.12160140470385, -1.7363532342870882, 0.6464275882844978, -0.7660338813837009, -0.09853776053048369, -0.05278905715262122, -0.5094325785313202], [1.2565123762307169, -1.1640143217760082, 0.30475225707786574, 0.49996790376496436, 0.7628819290877604, -1.1054271380110234, -1.2272647904737148, 1.0666843242948612, -1.0521201661232351, 0.2555060822208582, -0.44019361537982804, -0.7084310150363791]], [[-2.0139154344623327, 0.10421543873742012, -1.7843598942419248, 1.6811839960382446, -0.4905118761618503, 4.422540081323295, 0.9925569860691197, 1.855518954330002, -0.4351215292446402, 1.9820623138353894, -0.9700035869612075, 0.9785539741652415], [0.04062505456383714, -2.062892077967646, -1.8678936572127216, -0.05118364753663718, -0.7084000960127077, 5.191412694263259, 2.1964261147307624, 0.8550462377002263, -2.0387661876778287, 3.6839363106050653, -0.9042485158512596, 2.288409119101305], [0.3484846967806431, -1.1872305132737864, -2.2179620529317026, 1.4895623155757676, -0.18265838443166998, 4.8433003289928935, 1.731211518777207, 1.1351070020832328, -2.4700343259410404, 3.1602113283097757, -1.1786839513290972, 2.4320993712151506], [2.003317107770029, -1.6774045235574884, -2.0966160030135983, 0.01632917739823725, 0.8781014701801328, 3.6390310005821656, 0.35882704467350757, 0.921473058156012, -3.1541391253656386, 3.7223968799175475, -0.8182862436483954, 2.8589227991159936]]], "dtype": "float64"}, {"__nd__": [[-0.3562499901301067, -0.4437272610330744, 0.40681826684435873, -0.6649185619936864, 0.37650801309812454, -0.2141629276077886], [0.036456411639718266, -0.2842775841252436, 0.049247002409389784, -0.3223714176728921, 0.004090045824926773, -0.4845822179487228], [0.3492749515602048, -0.3829453935977099, 0.12375436118680835, 0.21433800051836632, -0.11262458189496848, 0.13330090690321922], [0.21560367478949627, 0.18237348371010323, 0.07708816622577995, -0.32224606302196473, -0.09681570269847324, -0.18515135403138186], [-0.2995608301126196, 0.06350931085149747, -0.23270654554205494, -0.23192555496129438, 0.20199488655185954, 0.591253446587775], [-0.12343343954258387, 0.06318650017411563, -0.1957076172032643, -0.057641943245632155, -0.14071255861049908, -0.45151433033240196], [0.06577684653293828, 0.1656978136658614, 0.45553003502753286, -0.246313248491591, -0.175917820516008, 0.08416454533595627], [-0.07924261013972288, -0.1977993745234092, 0.3741926804784188, -0.010583489947301525, 0.10315643839847315, -0.1539548779546342], [-0.0735401284875549, 0.02561591408977621, 0.09183125176916396, -0.307764116097619, -0.40847606251399476, -0.38745461944048576], [0.4275330918272117, 0.2337445604476777, -1.2092587513372195, -0.2188142853583714, 0.06198626447435918, 0.3399371268706233], [-0.07212838955545478, -0.256923775406381, -0.20567569277136513, 0.15366796392654514, 0.25466821128337047, 0.1720103519091123], [-0.004630867052527549, 0.21271680039453988, -0.0704373545928239, 0.6219247003714982, -0.2467764261807777, 0.17871444816809837]], "dtype": "float64"}, {"__nd__": [[-0.5430381441885492, -0.126246867015361, 0.19153676900468214, 0.37522329947564237, 0.24603541422528524, 0.7022019674545794], [-0.17301425193027103, 0.07343084683061, 0.19197064203182457, -0.21281234757193693, 0.058296362321486765, -0.26603147023893436], [0.061191072391198476, -0.20938202217050472, -0.2145487054194446, -0.28837004125949617, 0.4531839926387507, -0.38139136189510203], [-0.18545321248984034, 0.2541600933602517, -0.17490620048972244, 0.10906391067695403, 0.14684095108609663, 0.45878543468063065], [0.05669602734568353, -0.2832226608115022, -0.02621125457016188, 0.051671662542747326, -0.09305482514201824, 0.2615045940189724], [-0.36528088589240515, 0.29756642553081997, -0.032153470277453224, -0.5519553519183132, 0.2704641238792647, -0.3953711362809366], [0.27141120966568305, -0.06549945643683773, -0.3497807075435103, 0.28727083566803646, 0.0043742609203995045, 0.043964156586527206], [-0.2747398077018297, 0.298349295455154, -0.3825507241201406, -0.4910183328085489, -0.004117296590372632, 0.009506154683096846], [0.23073465924752012, 0.6676806130190275, 0.06212357054571641, -0.5555750853343852, -0.39237107197492815, 0.2604503783539156], [0.019061310702287837, 0.08508302786383165, 0.1750964265049116, -0.22056709661328305, 0.20852000527596531, 0.5712009682935527], [0.747794737941087, 0.1403560859312604, -0.2696037880934711, -0.2024336673711755, 0.02365805148431932, -0.005731306175411399], [0.013525608988596224, 0.06202920115759042, -0.17635177529648033, 0.13266866855976017, 0.14960572350237722, -0.4532886230711948]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.8336142351637735, 3.0640483454069876, -0.6069792232612886, -0.5381540377837006, -0.9832893680056832, -0.2154707809895907], [0.18227162781466852, 1.6545838714180192, 1.072041642173978, -0.501802935956302, 0.005312792188031271, 0.1574605718522182], [0.17753875212368, 1.1078082139198602, 1.3363253179297967, 1.5303006537724755, 0.8916654186392742, 0.20118123217208417]], [[0.18037625390313794, -0.1922521174998646, -0.012867383799147253, -0.1434135437646657, 0.18429693880101167, -0.3072365334206186], [0.6797906069074471, -1.9488682131564945, 0.8109579437473813, 1.2742786547823537, 0.37421221364618, -0.6790114444605344], [-1.7149296995475058, -0.16503215326838452, 1.1926234969172087, 0.09681820180648563, -0.6247556417057925, 0.8055386814312568]], [[0.31163679574920344, -0.6851646871464935, -0.7983474224001444, 0.8983596248240102, 1.2769237391910817, 0.22924719528072], [1.41654657170598, 0.1748332669284278, 0.7260031606986345, -1.231476414286833, -0.5741836591994145, -1.5649946948866267], [-0.5668961277708155, -1.1852913493094228, 1.6770560315789353, 0.11104440255393344, 1.217876097433861, -0.2685211984511176]]], "dtype": "float64"}, {"__nd__": [[0.3597283349532697, -0.5472890460281511, 0.3009787200766848, -0.11296622428560706, 0.3146931183508992, 0.14855733636549123], [-1.0046404968018423, 0.7936087374245964, -0.4566403413175715, -0.5682453488288219, -0.740658981826896, -0.17071576583883252], [-0.6092132452348539, -0.38436356985604797, -0.0006837967154505687, -0.2654336426634892, -0.2796989188985696, -0.15195580473557624], [-0.8281919037131555, 0.09025421073822143, 0.6195893401794379, -0.635107999973616, 0.11407027018138359, 0.11436119438867476], [0.37252326261362523, -0.6490758996406187, 0.420255862553977, 0.18586646652848027, -0.7875391738395587, -0.44962950277839475], [-0.09640867651941168, -0.010311475066454003, -0.18925818137766448, -0.6245276889083929, 0.17548682365821874, 0.5854726494608316]], "dtype": "float64"}, {"__nd__": [[-0.5503700617813712, -0.8555270766588321, 0.20342745911603455, 0.3176431744699129, -0.19985900931163691, 0.3154915441235901], [0.1523322389949252, 0.6463498181185772, 0.4277640841348658, -0.06344560896284449, 0.16362888452791505, -0.418333539963246], [0.2850996433234988, -0.18959171018757792, 0.4737338399106446, -0.14739381676354454, -0.5789652871227754, 0.4214767246268811], [0.1387109842957445, 0.312669486645706, 0.5774160836318885, -0.3316050580716449, -0.2712041383037924, 0.029230262911547972], [-0.39367802729625934, -0.7830783614830984, 0.11268877968844984, 0.37936717015967414, -0.2456141953348622, 0.3898056794234313], [0.1430575759963293, 0.2500508943473983, -0.1310202739637905, -0.004338257236061381, 0.4781099358816901, 0.6363157649873048]], "dtype": "float64"}, {"__nd__": [[0.6137487750527588, -0.1871634871510812, 0.01859583784479529, 0.2592083079279051, 0.48304177182416375, -0.036571852826145196], [0.17647293689626975, -0.6899812504511026, -0.11627874991797918, 0.31796830962013733, 0.22142874900619575, 0.7636149077247283], [-0.44201139550916785, -0.35173724307454585, -0.5433055132638555, 0.47653300941769117, -0.19971894849949862, -0.2995213954954918], [-0.19263890780763968, 0.019185966407690616, 0.04736341885110828, 0.2544798801440211, -0.5588158311171164, -0.0092902791363705], [-0.018735500395481396, -0.8881909436830704, -0.37361639217976805, 0.2052436933598065, 0.17248165302949245, -0.29433535115057424], [-0.16319284374137683, 0.14947328972025703, -0.8618979458167092, 0.30377541105306155, 0.68383675895952, -0.11083528602189319]], "dtype": "float64"}, {"__nd__": [[0.11042549579078918, 0.4788222524196785, -0.18648348574220258, 0.2729930387552199, 0.06939170192191535, 0.2659963161222607], [-0.08184350598371533, 0.3178940466712288, 0.1862956286999921, 0.06817808294224327, -0.16531878269398567, 0.4896991679469488], [0.009309197096354525, 0.39712530129625967, -0.5497665434380271, -0.031103411308944024, 0.0974673216103202, 0.7072707151905115], [0.359867102539188, 0.15310103249163676, -0.18879226656494053, -0.3759007733572499, -0.13667367803018887, 0.194838937191531], [-0.2187990347426009, -0.7741646022203721, -0.47327580274827313, 0.055017905047655635, -0.2708580106896233, 0.25582847264208275], [-0.46674276658935243, -0.2578584671268099, 0.6342000442909695, -0.16689210229681772, 0.704531311273446, -0.28831425085133244]], "dtype": "float64"}, 3, 3], "e": {"__tup__": [{"__nd__": [[[-1.2702251894776522, -0.9049997739849055, 1.1059449496819609, -0.4357083743856315, 2.1401998926611565, -0.7006408621293502], [-0.5458357120335062, -0.9595165167002055, 0.31091344451068553, -0.34798972569620773, 1.0369903712258717, -0.6609044503561061], [0.24003385743846645, -0.8754669403019407, 0.3774638705509933, -0.6997552458395692, 0.5085754692137685, -1.4318385183511255]], [[0.07757824097793925, 0.19425495272129048, -0.23291151468881832, 0.09606080964744299, -0.038933472954650994, 0.15883963159776007], [0.2614590011888811, 0.4833373077552746, -0.17883085826001477, -0.03570499938264389, -0.3600980589239236, 0.4039026311388353], [0.5626700505621497, 0.5828912094366895, 0.14093536078651436, -0.01842184309930006, -0.5410453889212402, -0.20905198099925257]], [[0.3887281253107138, 0.08504039013086467, -0.5163001128078982, 0.13205744073463432, -0.44546135139567433, 0.0007907933026994313], [0.1310810749242889, 0.37468274482320296, -0.5808325528456825, 0.18684048798194816, 0.0694749252027842, 0.39892296211089495], [0.3139402107659892, 0.008960987942759815, -0.2985500363880909, -0.06706407538492018, -0.19413298779583593, -0.2071637733851524]]], "dtype": "float64"}, {"__nd__": [[-0.5503700617813712, -0.8555270766588321, 0.20342745911603455, 0.3176431744699129, -0.19985900931163691, 0.3154915441235901], [0.1523322389949252, 0.6463498181185772, 0.4277640841348658, -0.06344560896284449, 0.16362888452791505, -0.418333539963246], [0.2850996433234988, -0.18959171018757792, 0.4737338399106446, -0.14739381676354454, -0.5789652871227754, 0.4214767246268811], [0.1387109842957445, 0.312669486645706, 0.5774160836318885, -0.3316050580716449, -0.2712041383037924, 0.029230262911547972], [-0.39367802729625934, -0.7830783614830984, 0.11268877968844984, 0.37936717015967414, -0.2456141953348622, 0.3898056794234313], [0.1430575759963293, 0.2500508943473983, -0.1310202739637905, -0.004338257236061381, 0.4781099358816901, 0.6363157649873048]], "dtype": "float64"}, {"__nd__": [[0.6137487750527588, -0.1871634871510812, 0.01859583784479529, 0.2592083079279051, 0.48304177182416375, -0.036571852826145196], [0.17647293689626975, -0.6899812504511026, -0.11627874991797918, 0.31796830962013733, 0.22142874900619575, 0.7636149077247283], [-0.44201139550916785, -0.35173724307454585, -0.5433055132638555, 0.47653300941769117, -0.19971894849949862, -0.2995213954954918], [-0.19263890780763968, 0.019185966407690616, 0.04736341885110828, 0.2544798801440211, -0.5588158311171164, -0.0092902791363705], [-0.018735500395481396, -0.8881909436830704, -0.37361639217976805, 0.2052436933598065, 0.17248165302949245, -0.29433535115057424], [-0.16319284374137683, 0.14947328972025703, -0.8618979458167092, 0.30377541105306155, 0.68383675895952, -0.11083528602189319]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[2.116279052253738, 1.4959723873535125, -0.7564013837864609, 0.4134403345603638, -1.6971879691720109, -0.8471942789869723, 0.4470211814039894, 0.4827335888797272]]], "dtype": "float64"}, {"__nd__": [[-0.07098904276411218, 0.42739143811241154, 0.13035222673013136, 0.6029160045782274, -0.17482284767398315, 0.1305318217145283, -0.2749560713447664, 1.1048034271559357], [0.32863216097434145, 0.0697937921593892, 0.3642835004842991, 0.48044552032664956, 0.26299092555630027, -0.7566993468971817, -0.36824632271476704, 0.40483438093948787], [0.049899913445503936, -0.13235399430870715, -0.2351406704159117, -0.07178538567436085, 0.3049605964092447, -0.45628976821069833, -0.5540055056292612, -0.1963974410941115], [-0.055813584265832106, -0.18670788512946232, 0.39456400752748433, -0.35931806612607947, -0.1508099782461947, 0.09283597223499529, -0.3216348673537175, 0.49381169962624094], [-0.4744095207041483, -0.17351743062505554, 0.06269555526376881, 0.395303713630328, -0.7753251677851842, -0.14782440049591775, -0.33811310660792127, -0.22969091882783027], [-0.12350508543756966, 0.23441032545297158, -0.19067450191107121, 0.09415521179928499, 0.6969763192533149, 0.5104545329644675, -0.14275036448408643, 0.30905044801237325], [0.048114217256664094, 0.5681256689237145, -0.015071418297643921, 0.01747973037961046, -0.28701833062487736, -0.3296439535752839, 0.49960565411991364, 0.9499501555899443], [-0.0066085851636869634, 0.058968237599994644, 0.3278487051646205, 0.6534920721227261, 0.06509550473533403, 0.12826251934093078, 0.14022371072629408, -0.020982241157556057]], "dtype": "float64"}, {"__nd__": [[-0.5727067561088933, -0.21288428464598808, -0.03237404597554368, 0.41431739181891675, -0.10161052819789698, 0.5118824934789847, 0.21208021746763717, -0.24728248743689407], [0.3009111086287026, 0.05422967558105096, -0.10966281466387132, 0.2362031713070754, 0.14345629519400613, -0.26052511394703887, 0.1324165165500578, 0.19787397393100614], [-0.3266473195136039, 0.0010317087364162477, -0.28797062759444914, -0.33280516190904863, -0.21995885673200297, -0.1276867750153765, -0.36459298638758386, -0.251963636721222], [0.0325317999813734, 0.0780907749758829, 0.05969898928651301, -0.2871634375377074, 0.15924652245270388, 0.4271835559560736, -0.6117690225291217, 0.4809830072422721], [-0.09195083706564362, 0.3104701568904904, -0.36196004318161434, 1.0260246921701783, 0.05616436482474653, 0.3323223930386649, 0.330635884274592, -0.0680523801431531], [0.8220999337072498, 0.3925253217513041, 0.11670212475983588, 0.8471562905099616, 0.11618493985418826, -0.5013587626772463, 0.5289783406469264, 0.07281372000148614], [0.07954990044002443, 0.24442695235256282, 0.3511815602450396, 0.17040881398467891, -0.01884328380794152, -0.049291551927666016, -0.4411807225793564, 0.0337403723840104], [0.1909313623945584, -0.2831233125981278, 0.323929583135857, 0.09418361006473312, 0.43429132877794985, 0.10914694884862108, 0.15567971601564173, -0.15905395942906475]], "dtype": "float64"}, {"__nd__": [[0.892222024483877, 0.039888366062868955, -0.2688518310396999, -0.5971318723860524, 0.25535236982365744, 0.38634185094834317, -0.5472042046950757, -0.49457564404195997], [-0.2342931167013898, -0.24471622036318103, -0.2822009363434405, 0.04603583487565427, -0.1208225523833477, 0.5266773350046667, -0.11789955450001344, -0.46428556146022804], [0.025859782679616968, -0.5138621796873594, 0.42212012413057504, -0.03245371313797858, 0.12462316090897425, -0.4685338923728723, -0.29379915313109284, 0.5622998733401608], [0.8012237900725, 0.3191324262886398, -0.5094749216198503, 0.38307963282137575, -0.30847186093061685, 0.46419413705862705, -0.6307396122845714, 0.3789073367842158], [-0.6127829310312738, -0.01575894262471327, -0.20848605285164187, 0.3606511043885383, -0.21053270409567715, -0.18928312291850413, -0.6455338693175849, 0.11162482420336864], [-0.21935112463549672, -0.04161700045579764, 0.5367030952786759, 0.20312114789895164, 0.6358648168175662, -0.6526100327706232, 0.20445461503768728, 0.30943297308052053], [-0.3572116216319545, -0.22554312603502022, -0.0614041606672093, 0.3006986703249395, 0.6869754445106384, -0.2746353740207221, 0.5210023515338802, 0.15794788087899056], [0.6970367961878338, -0.14457543682426618, 0.2547318394044885, 0.4926772513549724, 0.2706267157101124, 0.0035198518184686636, -0.33565013396612037, 0.632218718901317]], "dtype": "float64"}, {"__nd__": [[0.34902964346734894, -0.35898061594227854, 0.1048039976990409, -0.66924669246903, 0.48490431122874045, 0.5920033396805117, -0.7956596263936744, 0.1459432373716095], [0.08973619994714391, 0.16947692981655058, 0.8661947298467974, -0.2141025354479067, -0.2049569095528465, -0.2699431321221163, 0.25842843628208007, -0.3956342782956337], [-0.8691781244997103, 0.06174341562651611, 0.8773046373178728, -0.3712379111650918, -0.17507719412143447, -0.0940445403491827, -0.31787673416245155, -0.7146589511646944], [0.48563935196322305, 0.16602319808544397, 0.15777799434658055, -0.3971966906083385, -0.1830326397080485, 0.46492097956960254, 0.5444370168755379, 0.14284555930915985], [-0.8096363822889588, -0.01761792833622355, 0.11544248431153586, -0.02679668654422886, -0.2228584302270217, -0.2646314733771237, 0.33131623283970857, -0.0022918209935243942], [0.021214309227905948, -0.04220753142710559, -0.5123341812507987, -0.7610156263868443, 0.36986122711036273, 0.6108397225419786, 1.252211065901482, -0.18324664131261761], [-0.1147867444599669, 0.4811080073890785, -0.016823209655124592, -0.24074474349224959, 0.25243861976044174, 0.19503413937422173, -0.4040830555703907, -0.4387483425763519], [-0.5459387566791158, -0.9930599585260582, -0.41990859845889816, 0.2429619040625553, -0.2334381587738105, -0.6405726862867717, -0.3061162611298578, 0.3870536062823763]], "dtype": "float64"}, 8, 2], "e": {"__tup__": [{"__nd__": [[[-0.29606823916818786, -0.1150373540829555, 0.04299127575972115, -0.3421945624401953, 0.02615414081246183, 0.0541622569827273, 0.14845630279919944, -0.13820013915381169]]], "dtype": "float64"}, {"__nd__": [[-0.10091192372787709, 0.09376742382795772], [0.12042028521323941, 0.0533054179320078], [-0.23659785007017137, -0.24105056371404635], [-0.02921046832348452, 0.11391101578048198], [0.22064599220335268, 0.1627675654987126], [0.5446209176820879, 0.05415455945633862], [0.21139180675557645, -0.1188937964827384], [0.08148031074925519, 0.135016008553287]], "dtype": "float64"}, {"__nd__": [[0.01653167178024842, -0.10002140699125875], [-0.17879360963308927, -0.04408258333473061], [-0.024583996503786485, -0.01885250281370751], [0.24849023189066632, -0.02402749984308636], [-0.11909420552977265, -0.23343121803209943], [0.11971402952158329, 0.1242855930412877], [-0.08586505950231114, 0.27282257572569674], [0.3249676125307571, 0.14267878811594442]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.2861168467772313, -1.0467070894040333, -0.07833133593073606, -0.9698466807348396, -0.36828994766380324, 0.018690241876018508, -2.8860343498292282, -0.4456269412297261, -0.7856209416175974, -1.357026298190909, -1.5824147388844243, -0.02656175240655801, 0.8682524817114011, -0.3210275323064348, 0.411557839392154, 2.4823694202268003], [0.6064878192170419, 0.06968297764347782, -1.9044036436813523, -0.034612515817315254, 0.2592126503271619, 0.6281262099211232, -1.367850119842656, -0.6018540302298402, 0.31101254546923895, -0.09225915410890131, -0.3735489764728391, 0.015216346443335598, 1.0296120226712269, -1.0399103778887084, -0.7029632024691873, -0.6328488407353668], [-0.46474668209703085, -0.5369998680689705, 0.7353263444458638, -0.2659291792604864, -0.08872600545157111, -0.8760994347763693, -1.1840209286456063, -0.9488844878059953, -0.5109367864574833, 0.4281615259767058, 0.10928857731262899, -0.02016467325578706, -1.076070510644276, 0.4066710459828126, -1.3532478004320625, -0.4106804363508371], [0.9940956803698386, 0.6191777204190454, -0.3419987609889955, -1.0447753232268617, 0.7389502139788533, -1.4894454290136532, -0.016183357203103862, 1.044115522007898, 0.6117901602719161, 0.47070419138207037, 0.42562698895068696, 0.2771338949336642, 1.2721148198299503, -0.7782129125096836, -1.5538718161352985, -1.1553003962529576], [-0.9242016072329825, -0.023615521431508007, 0.721699403802288, -0.012427918342169386, -0.43322345685680075, -0.403628939282468, -2.2771060648084775, -1.3410927777213906, -0.5138042447562108, -0.37170624895656335, -0.5929147936497755, -0.5725382924386921, 0.7595464221779153, -0.5373963439887024, -0.609728118326783, -1.4565340505265953], [0.4963442351275113, -0.6083073739314061, 0.24130015918524983, 0.8609235338295294, -2.06133288612253, 0.304903692754087, -0.5722832435411837, -1.6500189432946057, 0.4526545754562698, 1.005171873187044, -0.14677086082128982, -0.6508864537705495, -1.0867490827682889, -0.7398656406793134, -0.5023251536907779, 0.3572714981128911], [-0.7064923134766156, -0.2687511707239277, -0.16658017660612578, -0.7178174603229281, -0.27067140691955877, 0.8390399159635948, -1.9929955247682323, -0.09571158560010265, 1.2219905944249931, 1.8827859522373611, 0.09194642115799362, 0.63903183581952, -1.2887875044418091, 0.5012468515032517, -0.07749587473464413, -0.6230360021431515]], [[1.5684506154823512, -0.8484800406709972, 0.49886093579570234, 0.12636453585554416, -1.3895232769524966, -1.6453279467430237, 1.26218718399883, 0.15451302345837017, -2.2959359642708366, 0.35650770777545226, -0.14760678094103, 1.8867511840444564, -0.04954125588878965, -0.44585188381922464, 0.36732890427376436, -0.5221341306575006], [-0.25763715109169577, -0.33951374081379, 0.5000248788071376, -0.21930311642212708, -0.3422897752171773, -0.11938902801367812, -0.38504858831184974, 0.3872256078804257, 0.7274322305957461, 0.34476890948055955, -0.2127624700345277, -2.804978226631951, 0.7012112445062162, -1.7640773473437477, -0.04892786272638775, -1.0385538234565936], [-0.8009041496940892, -0.0627707434076035, 1.1026207279900146, 0.18246401917685462, -1.345813767874061, -1.1236840794834997, 0.8635172861537407, 0.37000413904470736, -0.41585929934517507, 1.049044493303651, 0.048030951179065515, 0.01730026856436055, -0.4045215799110555, -1.9212331538018792, 1.147151460839432, 0.29912625836366674], [-0.15309131140171173, -0.024638488144993222, -0.05865208119280461, -0.6343912772668776, -0.5579099843539568, 0.4307502241086579, -0.20651485614199402, 0.08349037660122326, 0.008961433935980102, -0.4329810567655977, -1.1398810218433386, 1.43735489665741, -1.3303825387044272, -0.8115243647359092, -1.3696245237741378, 0.2922646645221113], [1.449870322462048, 1.5754558383493453, -1.270341771411381, -0.488838978866329, -0.8276026785094518, -0.047088177578829285, -0.7819859276837989, -1.2324787961144728, 0.8452898619567069, -1.5100554804469482, 0.9406841442970384, -0.760147115313614, 0.17992662944963494, -0.11115870680445104, 1.1256401102171578, 0.44025205884660445], [-0.6230039999735031, -1.217401625430646, 1.46336487091788, 0.6072017008989987, 0.5468044659580632, -0.9863866406923117, 0.33273828462938027, -0.3396902572317273, -0.5423207111875198, 0.6628072406215973, 1.045902257832602, -1.3292165821070459, 0.7394420264461933, -0.3684318156750952, 0.2636033981992316, -0.11262361444572747], [1.171153036524932, -0.4213645041447077, 2.2325109042556113, 1.1296468618134972, -0.03143020475539813, -1.2438042685077302, 1.009706125314196, 1.1098830821469698, -2.471906941539646, 0.4706376542758209, 0.3371458006552597, 1.7249772081005899, 1.6438864319338111, -1.3029426037095733, 0.21632500643556962, 0.1550244358653398]]], "dtype": "float64"}, {"__nd__": [[0.40662604669327895, -0.802462494368302, -0.14953579731850627, -0.11907368086829975, -1.063037092764843, 0.20994156212257076, -0.014831580453275593, 0.1751741710398218, 0.2710291761228451, -0.18079344970534605, -0.3542883663818623, -0.07547284395596927, 0.07614247556256903, -0.17467566579581006, -0.29692471882293275, -0.6126512194825364], [-0.25989283300787824, 0.623691067834685, 0.6486363669347539, 0.21243094717627042, 0.09242614286771651, -0.36340743622574495, -0.4099784300549776, -0.338144574801571, -0.4772718384688333, -0.19984239129584813, 0.047124325457554794, -0.41584542055819035, 0.15034469437942388, 0.039238396507087304, 0.08457630847137568, 0.4752672013575989], [0.2212948535035396, -0.6023067750028114, 0.029164236440577886, -0.25361633728498734, -0.14683428447138247, -0.29195055252146734, -0.7192888904718078, 0.4141727904105665, -0.28964104410216385, 0.7944881875745352, 0.5953250633278742, -0.3474287508085925, -0.7431258334907788, 0.18083356549156226, -0.03824617217505009, 0.7884778178501735], [0.13867769833522103, 0.07560697511207816, -0.3585892376070781, 0.11200763005820302, -0.5428180934359574, -0.43048481293084423, 0.222308631907264, -0.4532591012733814, 0.32875776161002496, 0.5672052551172513, 0.011762892951894472, -0.18700820357469214, 0.4596587598868074, 0.2861221752058122, -0.09779383631001762, 0.2737097805517285], [0.3324277163923839, -0.32707747218536953, -0.7454308177188982, -0.042350765217761176, 0.6849322971509202, 0.7633560789782528, -0.3203994760729292, 0.44017325478561287, 0.6347142204752041, -0.2279568527362813, -0.3923831537484223, 0.03853794092768476, 0.41715243713158556, -0.2838177125416672, 0.8834282096501982, -0.18451030352857223], [-0.44168860994712533, 0.3241529773439267, -0.003404442831052233, 0.47855926594520926, -0.46590533400570866, -0.19708517769519357, -0.021455139151557376, -0.142608242965981, -0.0633280302370838, -0.284997149486081, 0.0310523645614251, 0.3797191513321607, 0.18600491787088566, -0.49274128302877984, 0.012354495655398508, -0.01841096936491898], [0.055445814747882585, 0.19676374449308012, 0.25382375836363485, -0.20134449553204462, -0.3891092916729644, 0.2056595725909594, -0.1650474586521144, -0.004028674985169652, -0.061609983054924926, -0.5074156226300469, -0.041345376067872586, -0.5257873627268179, -0.12277087261428608, -0.004815560945579136, -0.2741681997141709, -0.4389080086081718], [-0.6267286580148838, 0.09681028934971836, -0.38037643142959965, 0.8037115658992667, -0.7478238467234447, -0.09205487371202288, 0.2620526923257585, -0.5252612022380592, -0.23991344108594462, -0.5404738897420394, -0.20586526311952139, -0.17916807215290834, 0.640821149439037, -0.4319106311598928, 0.19303605287357106, 0.7360630733340452], [-0.22777868789506314, -0.3888548546614285, 0.8051162616079129, -0.16073605008520112, 0.9561960629722828, 0.05718612133579856, 0.09450965075711719, 0.36855018756907154, 0.3583664053378971, -0.5237206523152328, -0.22903134922027021, -0.3630301628609975, 0.1921714794596987, -0.5314593838206306, 0.5521793163702576, 0.22751859484686096], [0.08266152443338304, -0.24753449957769202, 0.23067936253803312, -0.34913963967980005, 0.9267303523582395, 0.34424185476007235, -0.6249870179701384, 0.2359800586310152, 0.5042715163123517, 0.05940302185224016, 0.9385481201284485, -0.26409326767131186, 0.37391350501176424, -0.2670550700778935, 0.0834007558360967, 0.34208834569481045], [-0.14398763198256181, -0.351181883922546, 0.3522056390261383, 0.3812622901574795, 0.05206070457120951, 0.49144829691115294, -0.6806116119209957, -0.6730566855478114, 0.4640649076651471, -0.14295596730466942, 0.07530751473933386, 0.2870279848332218, -0.3448979722331436, -0.6779700618671232, 0.42188889357959874, 0.22689626811797112], [0.15674129115075014, -0.05912375723705572, -0.16184840954602153, -0.23926873068478027, -0.0622357931478191, 0.6799847950893967, -0.12516174846963132, 0.010147575670584982, -0.18046687580720355, 0.6383441789727815, 0.5383947596717052, -0.3921779842338521, -0.06894162469208516, -0.5172035835558665, 0.06411470046623295, -0.3329281223024694], [0.13583359407767534, 0.47788775262442207, -0.3332116360014298, -0.37453530223789566, 0.0217443489716987, 0.14214138312326727, 0.19715711203585823, 0.23742693659512334, 0.23045555998216313, -0.1491352200874717, 0.061196926163822156, 0.11809721208075741, 0.20035436513542587, 0.2703069386575345, -0.3695864908550887, -0.42771872221243756], [0.009487281808875266, -0.15121205856460132, -0.06216123373488765, 0.06943767035497385, -0.2795321603957545, 0.38697613921699553, 0.4156402101628568, -1.088535811474985, -0.1871881126999771, 0.9477489454627479, -0.5504488378730509, 0.6164257171695272, -0.6603051357319156, -0.23475863835754274, -0.18949110830922164, 0.38474966823407253], [0.05684720657475752, -0.4273400575040226, 0.14622563768892502, 0.07560656318097918, -0.8601011837070965, -0.25246996583112646, -0.03431133738406813, -0.6261852777543375, -0.22461802871849681, -0.40704740224604363, -0.5165117489913653, 0.2820075887166162, 0.6056034058644779, 0.07378260283143458, -0.36133865971137263, 0.22114705746402247], [0.4462336659992415, 0.10060850111352733, 0.5155734311337968, 0.710392663421189, -0.38992134829203556, 0.33677925149787696, 0.5214279679532227, 0.5634768263529367, 0.15433994302602871, 0.3334905849308834, 0.6129916753762115, -0.23867426358000327, 0.22203035658931905, 0.22384949333056695, 0.09170814350984521, -0.5253789310488803]], "dtype": "float64"}, {"__nd__": [[-0.2521591378249368, -0.19571102307163032, -0.16129624571325932, 0.14127303698854604, -0.5243466268162412, -0.6702628128662639, -0.5164768730324856, -0.18971544362922366, -0.3218039272030271, 0.29775346920895657, 0.24892048156860958, -0.4658701849251201, 0.3413150529513962, -0.20411802321317662, -0.6141423122587364, -0.08690803186899632], [0.013670775173785787, 0.18159479537772882, -0.2566706182892797, -0.5009031401704819, -0.37931047814189717, 0.5962984065983243, -0.12672577765361184, -0.19089403427003734, -0.0039493039650860635, 0.3215611347886905, 0.210109078886769, 0.2565556354451732, -0.40806629978626924, -0.17245225179800122, 0.5458872532991029, -0.45000489256473586], [-0.1597990615687265, -0.14219632909316635, 0.4128490043315344, 0.9138482425309498, 0.6336470453198917, -0.29498731400127054, 0.2050435106939304, 0.18026917177341104, -0.37525462046903124, -0.09978342175213063, 0.004916951185915973, 0.37997434238664185, 0.720485922611284, -0.3140552498877227, -0.04975884133266567, 0.06690519606494515], [-0.19440440811290213, -0.7675144804196223, -0.7394241199355208, 0.08267847308888682, -0.43213275824193814, 0.4059640389118888, 0.4673460712085398, 0.13709355866055054, 0.14188687863371144, -0.016230776878358995, 0.2326588651205384, -0.15298972085877272, -0.1830245512498669, 1.0416851118614125, 0.09412806978051112, -0.34412866172405315], [0.0667211433911656, -0.5927509925685588, -0.09322515143054677, -0.11161554174535011, -0.24607281623023944, 0.7456749273875355, -0.04672854234877418, -0.04303563587891502, 0.08946323383886697, -0.673057336064534, -0.6120816684436003, 0.05216419016557059, 0.2932441922141769, 0.6440715392879761, 0.23565809729175974, -0.15080917910383662], [0.02249663209515583, -0.5803745984426208, -0.09894075139255465, -0.41628685035194074, 0.019016939670992165, 0.02897578714269824, -0.2139305921665789, 0.3888640679204738, 0.27403235209512905, 0.9378561385186738, -0.5053221414231444, -0.3189606548550693, 0.6814290789993681, 0.3798918387322176, -0.600065328727069, -0.3881216611933804], [-0.465189996262211, -0.30577857269557773, 0.2283186102101484, 0.027630271969861647, 0.5201445833368177, -0.313011977308457, 0.30042116172422356, -0.32679775521403415, 0.24935420380512163, 0.7346584167016017, -0.18678436618929958, 0.7881553989155349, -0.2563742441031712, -0.4218705519256199, -0.30796560356582126, -0.5629424752894956], [0.07999464011943813, -0.25779648369540603, -0.2582993505206031, -0.02221244834700302, 0.04976326681777379, -0.027011032601563397, -0.6969356107605428, -0.11069341320364684, -0.12600758476411736, -0.29100634472921577, -0.6212756054764027, 0.432768672806239, -0.272218954888798, 0.05877269013279758, 0.19069618236030933, -0.3821177265855615], [-0.567767882045655, -0.11446677600736994, -0.12174825465717942, -0.03658106898813499, -0.047359537388776546, 0.19466309471949167, -0.22088964914247053, -0.46209235173656404, -0.09339615234555182, 0.0005428194703054798, 0.03984244104095855, -0.18129400560968245, -0.3207019924990906, 0.19515255336839699, -0.4678775909384372, -0.3160240851266438], [0.3480477100982211, 0.11951713945982892, -0.31951467568273634, -0.08653521187636654, 0.07798332809649351, -0.15134766147135825, -0.2521739414481538, -0.3236107876004597, -0.2901409652237396, -0.10002964014196175, 0.023019639757067927, 0.2883687309406663, 0.10938046099988502, -0.09877325674610248, 1.1370218568591057, 0.5307929931176648], [-0.041079196694219444, -0.4326821454216152, -0.5515037281827232, 0.506620247745989, 0.5850156941299636, 0.7081877265110803, 0.0029740004486249606, 0.4592978784467677, -0.11745887289478599, 0.6536450705428751, 0.4308939490810428, -0.7324050403806761, 0.2087259695163103, 0.5659982201841575, -0.4721133071582002, -0.4189181176869039], [0.1559044508489896, -0.5513528519610368, 0.7106658822601298, 0.5549051197941552, 0.24061395502402944, -0.6999151986290829, 0.27933129266637197, -0.5063319989061087, -0.4468846674503474, -0.28942064650134885, 0.19196558820436926, 0.03654310277281383, 0.3752464397170916, 0.6955906427926863, -0.44066186040409916, 0.014322756143177207], [0.4877207586620115, 0.23870821476536977, 0.3662219014515231, -0.49490575684103927, 0.33687447921835223, -0.07600892781864542, 0.1608862782466168, 0.13932920815150387, 0.010963164524733187, 0.7042542818232127, -0.4664175781496407, -0.18959492935741767, -0.2461581647474652, -0.6035687048186001, 0.17956136843727133, 0.012847569063608774], [0.48397147023015163, -0.4267856428183991, 0.2764528504140895, -0.5065779321489388, -1.1375076036032912, -0.05863054830864465, 0.7081623721217586, -0.1862832208289181, -0.1315077019941516, 0.10120286173499293, 0.26038745420105547, -0.4723034390214049, -0.031082967580993787, -0.07642992397896864, 0.2738470492372803, -0.10472235537832893], [0.18061609984164761, 0.29732216637147024, -0.0018059112338080282, 0.09475197274217889, -0.12973207253626212, -0.27747157955359864, 0.16747085352263372, 0.21933311581229972, -0.2632282162183406, -0.16362469503331256, -0.43559598039720293, 0.450974379088849, -0.333743792271273, 0.07610711276850249, 0.13298229148776658, 0.8308866091269631], [0.39433521407182837, -0.05088503303316747, 0.5634194208755167, -0.38811064250243293, -0.20756085088166243, 0.27407133860892935, 0.15295500131622844, -0.021506882333642004, -0.4370430189404522, 0.014200226916296927, 0.5263365480474421, -0.43663642695846544, 0.8594976286531906, 0.31390549263166134, -0.4546298347266562, -0.10675857463321006]], "dtype": "float64"}, {"__nd__": [[-0.28895080852154514, -0.15701934183874477, 0.5588451958670143, 0.31832749357639334, -0.4218588740451088, 0.448250114538584, -0.40440251817343614, 0.15480934229542406, -0.2657660870039747, 0.1746804005739269, -0.6400803853291339, 0.16576475922536582, -0.0726654460375914, -0.0885280174370441, 0.5803624689827096, 0.18151262661160886], [-0.009223682862042837, 0.026770818745441127, 0.014318327052990278, -0.6820957535780522, -0.07425391167116861, -0.8365558573435203, 0.8178219276901955, -0.04784474550941365, 0.1300162162251133, 0.32212765293972756, -0.6677792697263195, 0.05526296633129152, -0.25335840715004526, 0.6859087097644608, -0.0435699558953615, 0.4056877111167455], [0.1377901507651207, 0.4709896647076538, 0.3014978756188116, 0.2003081606353807, -0.06588209063011255, 0.3460443153161613, -0.0127474399011584, 0.503670470575666, 0.21484355178590367, 0.39669470506705334, -0.027489686413424877, 0.3301296927927366, -0.4304827559747499, -0.6480207422842986, 0.3486772881068349, 0.6395819815629735], [0.6158796720441727, 0.13347442697563858, -0.4907751118128835, -0.26414664325137693, 0.6393919356293062, -0.019345327001608098, -0.16743602201204516, -0.3861082617333524, 0.10975293325222575, -0.2230497443113674, -0.6725012899993726, 0.1466487030933806, -0.02843405363436707, -0.21272045604453027, 0.08926923377690786, -0.5397534608050039], [0.036102629927383265, -0.36924998558949973, 0.11493436473267392, -0.06118263905434898, -0.5263620171480986, -0.08294303009020079, 0.19130226576301573, -0.15356809169807378, 0.2721785073493956, -0.39575254651538105, -0.0037166630728264483, 0.5150802913206828, 0.07054901984886307, 0.23703621374319095, 0.705069956369718, 0.02631445698943521], [-0.5531468468752858, -0.11386293086539528, -0.3014625022612159, -0.08193162928685799, -0.19320009284788628, -0.17282989167398982, -0.07108728577341854, 0.6387979351445932, 0.07000269701179594, -0.3814233187589471, 0.5230289680998798, 0.6099088916253611, 0.020953397359566747, -0.2422269903645743, 0.6492860046262633, 0.2448290403806908], [0.3867930544185243, 0.05589253861809793, -0.3997425168593543, 0.6972430916231076, -0.018059253778712692, 0.18678273660130185, -0.699327095989326, -0.18170684473174464, -0.13920433087765768, 0.25391462505448564, 0.3279235781896179, -0.26131619680306517, -0.8587743344329969, 0.12606815364841528, -0.12394302554482235, 0.42630009293887405], [-0.31271821218482687, 0.4305979299803089, -0.46481709876237337, -0.3273118505624226, -0.07674383197465784, -0.014083194368879818, 0.36658380551845915, 0.06071845495095705, -0.7341415083073363, 0.7586106712221342, -0.3044722890331274, 0.2713273762418769, 0.14625127860604234, -0.03946163025335544, 0.4005442900778034, 0.15537138479428841], [-0.1562600652995828, -0.6712787206059658, -0.23788460343044068, -0.2644101111687678, 0.019704748514057038, -0.23899217428652986, -0.34978204295546633, -0.7022184798104778, -0.46997284357222785, 0.19377511865308691, -1.182716093907875, 0.07923231063161928, 0.35746616529542247, 0.12109251240860448, -0.13033124229228485, -0.42837079413448365], [0.12994921772203014, 0.39916772310496484, -0.4738229901368324, 0.06416744646924948, -0.3475455065215636, -0.2967003935759313, 0.050452496646215594, 0.17785758729315496, 0.6795570066050747, 0.1872188623614457, -0.254733107047064, 0.1523136188537466, 0.2087993013628887, 0.5663826351255431, -0.6841498310471453, 0.2775199770219427], [0.04463569295494119, -0.4127272946447096, 0.7081284237228237, -0.6947375065806124, 0.10794861686055424, -0.006793179291397582, -0.026283600090023847, -0.16461812202134263, 1.0183851491363007, -0.4101863267438992, -0.42138957331323124, -0.253860879418241, 0.5675501244706682, -0.1253021690303396, 0.4587656863131585, -0.3142897895098169], [0.616388104104773, -0.29960836900975324, -0.3639859025034324, 0.02592675997853047, 0.5431607031052347, -0.19770794024357732, -0.07771567686836392, -1.0277679519425633, -0.06715373598742322, -0.06696328549564527, 0.6249001378138117, -0.034669369206122694, 0.38918621346061083, 0.4932709623561475, -0.006798199449539437, -0.49750192434378004], [0.27581396493112076, 0.23188452003852833, 0.16982974851325452, 0.3215709685189591, 0.893469451167304, -0.10390758373389775, -0.3926318372686477, -0.0520078551680474, -0.24894493428874753, 0.2765240585031119, -0.36731575656303656, 0.5112158929175485, 0.02582984145796134, 0.3707473052199235, -0.610196338749422, 0.3497044843622987], [0.32007715793609287, -0.5197564540441499, -0.9281048650105609, -0.16237072179723633, 0.14221277874228153, 0.18181280228294008, 0.018451059346750117, -0.3222008459839946, -0.5741336186231548, -0.21698566449200263, 0.9082574451080124, -0.132245506462515, 0.1374671357964408, -0.17478037555274564, 0.40703017362369576, 0.21081399107770157], [0.01403311675835027, -0.13702005561779004, 1.4754236708389834, -0.5140611645151768, -0.40083059280791844, 0.2342447238412937, 0.23716601619748234, 0.6897132650955875, -0.057625394794136614, 0.5179191765129011, -0.603663441745871, -0.13627275124144528, -0.46785375117949857, -0.6416924502420229, 0.30206676039818436, 0.36027209539632454], [0.6150541248054104, -0.10138928309605433, 0.4851148898915316, -0.10414594223373881, 0.06256113595236887, -0.13787538984862988, 0.24278394508663315, 0.5060679958937299, 0.0730721566440992, 0.622773083615201, 0.27227158201749696, -0.3354981499804766, -0.5611650156949203, -0.13903976024425477, 0.07222114282943015, 0.3928467129921425]], "dtype": "float64"}, {"__nd__": [[-0.5588981123279625, 0.4742936130426858, 0.1454664103048851, -0.13429135464711336, 0.15902391471760585, -0.08390574517016632, -0.14883371660403455, 0.3264354182297291, 0.44851410526209384, 0.2865177810762271, 0.5718625670956338, 0.13976306464007981, -0.3087479511679053, -0.21418077187149376, -0.07713238362439725, 0.009186177956991774], [0.05762657058708022, 0.08148892194151276, 0.40901782057848823, 0.3737501272679646, 0.3437716137749348, -0.4704782902456912, -0.2206550839257695, -0.2877921540381371, 0.029560584765853666, 0.017705412267440445, 0.44597187969977964, -0.1989373744276687, -0.5315532956339949, -0.1684533708517636, 0.38598674897859936, 0.21851729538373715], [-0.37736557885235333, 0.9175923563467624, -0.29681437042576686, -0.06256595351177684, 0.06254857005458234, 0.35060137998168095, -0.2771958510534781, 0.09852504066542449, 0.19613049885027722, 0.32589313985355156, -0.6493410193535674, -0.23432560470218278, -0.3198624727618696, -0.4661832789749421, -0.318123819660719, -0.12842557655091144], [0.2936711454419994, 0.1860351188854723, -0.3041615231486907, -0.19039500320785485, -0.18570696858944608, -0.16911673915270187, -0.10455000363805966, -0.09405410709338208, 0.1913154946551289, -0.43096878286348633, 0.32457983394251144, 0.26479383222642455, -0.3975076446481951, 0.15623178932320492, 0.13189985051661726, -0.1580674738126136], [1.0306508402775039, 0.16277397637741148, 0.20808902694827422, -0.025293425178896337, 0.8359472495906991, -0.697425324890844, -0.13149826031765577, -0.06516696495906096, 0.5965411071572269, -0.341925531957421, -0.1428462256195868, -0.8986148790636213, -0.49467050364030174, 0.31351438769497525, 0.10578920032435402, 0.08165546043735035], [0.40732530260707067, 0.4427978349620827, -0.5449495699940152, 0.6806191327348339, 0.32272557271981517, -0.10154660564463458, 0.027935565004529206, -0.5134712407956015, -0.3104112832653043, 0.11531892191377253, -0.0494769996423902, -0.20919435559500982, -0.34322189774143563, 0.002085756121620297, -0.24795760351945845, -0.2694079010817931], [0.11346280653280912, -0.27573498379993666, 0.40346511610917457, -0.2457177905607099, -0.5719256354730343, 0.4505791859712031, 0.3929542371073583, -0.5215779406723671, -0.09392084076855911, 0.04203634671448377, -0.22289483611070204, -0.6003129777687418, 0.4500363832383239, -0.2131752947035487, -0.16252127829350416, -0.21934476207592915], [0.6655846635784798, 0.36585691282516764, -0.5036662913902843, -0.7017122546217314, -0.8293626880501285, 0.27040757545741173, -0.38269788844965075, -0.3115798331767503, 0.0463278982323205, 0.2052306032675647, 0.23440159231581204, 0.36480248221244116, 0.8112787448019303, -0.2365039882339579, 0.3460463855076079, 0.7869499040859012], [-0.4623628536147688, -0.31851399560706173, -0.3500862149570547, 0.28102828368843585, 0.46869113744836693, 0.34493943558490386, -0.060555568506988934, -0.8386824213225051, -0.22707452926444402, 0.1996142020430649, 0.5391520313789164, 0.3378965129391204, 0.19914098274552916, 0.7888905472190009, -0.45618695709739765, 0.23144108440861283], [0.25733357824646147, 0.4810477174614438, 0.4024605087101363, -0.07838308708741651, 0.5961216790664048, 0.2216897906596992, -0.5849734175260711, 0.569641909920445, 0.22738793545791292, -0.041508672356857346, 0.4486449701838624, -0.16630565654431734, 0.1322975614859497, -0.5451128268742552, 0.03826367183618471, -0.3352146376059609], [-0.380857620352468, -0.45186060693704566, -0.1420565104424601, 0.3071858781731627, -0.05289428751518008, 0.061843794419538446, 0.03462301018470666, 0.2561011097535431, 0.3675795367682817, 0.13598816460979402, -0.07659549478840967, -0.2018586640402371, 0.37858997024818986, -0.07244424588517379, 0.4798517124864562, 0.6024623183873055], [-0.30083815149498905, 0.31758978362866974, -0.04232742248101978, -0.29659386922191516, 0.502999035417552, -0.09561547225839578, -0.4335336360242565, -0.055273195932890706, 0.20809290313801176, 0.10065429069230568, -0.18526608205977754, 0.08257471678383382, 0.18174284448947828, -0.0955573912127599, -0.7424567616344728, 0.37166315923909915], [0.2060703257590699, -0.7576864187829923, 0.18400775506981953, 0.11842906660562083, -0.11632106504340226, 0.23103789606969405, 0.5463968400865783, -0.46832249248342556, 0.446278542301931, 0.06681816158515484, -0.19884851039755863, 0.11660318738770817, -0.06327818935874406, -0.4666466787679383, 0.05378414027324411, 0.6220555689481275], [0.205181538007682, -0.22972913869938633, 0.22191682083985814, -0.11166026063105941, 0.018622582216292035, 0.07793577908912636, -0.36926963293968634, -0.5480997757377778, 0.013869237163976889, -0.41838042853236956, -1.4438168506440774, 0.46864013702786417, -0.39774443070607896, -0.05818371432748124, -0.5106267338004529, 0.4931092382759228], [0.11131035180876542, -0.008327370299508051, 0.46311382258669553, -0.4718088609366815, 0.40793118051286137, -1.1305190113889685, 0.19182460988720323, -0.09184723319288395, -0.6917785166388661, 0.514646297068919, 0.01491345100944696, 0.3415920602517849, -0.4486663425817449, 0.2566549975785292, -0.25812365356036493, 0.23652935747257187], [0.19102677315012956, 0.4575219691503743, 0.2944484632151758, -0.12381627897271522, -0.7132156505331855, 0.5031428728292465, 0.5482180941411237, -0.02518506015631147, 0.09506191000225922, 0.13158343709863174, 0.6748171651049131, 0.8401760362219302, -0.057743010375223225, -0.007808359778672768, 0.20318009537999135, -0.12918807182197728]], "dtype": "float64"}, 8, 4], "e": {"__tup__": [{"__nd__": [[[1.0681087527097233, 4.051323240838898, -1.868850398396661, -0.3814738058366247, 1.004259490349239, 0.5064225497209277, -2.0535139712844668, -0.9881559000930112, 1.7191862839550995, 1.334249343662825, 0.7697451863864015, -2.844017551878097, 0.3380180554456238, -1.418832711981951, -0.6042035349946558, 0.39561310403709593], [1.705369024424966, 0.6924498982284495, 0.2989890238829641, -0.2839460558057778, 0.805947733509244, -0.8424057433843702, -0.04664550785710527, -0.7138151853738328, -0.0850833418252507, 0.3762893605320863, 1.7086751757998018, -1.281200988812754, 0.27709991341796214, 0.3509760129328521, -0.03107125734757204, -0.33208698219594207], [0.021172941636344422, -1.5261001232350409, 0.5002336293933539, 0.3652131534810258, -0.4172805341922814, -0.5691866094764758, 0.8170106241474009, 0.026881937184483663, 0.6195799979931277, 0.025766726335637823, -0.06394903239855819, -1.0849362859120648, 0.03126981905605891, -0.24330030173425138, 1.100599647485032, 0.6393345629452536], [0.29511790393269843, -2.26615362274412, 0.4591986055738599, 0.5145806234571829, 0.5441708790733304, -0.2676105386555105, 0.43713231903966615, -1.848302849990676, -0.33248338290719426, -0.4455379715445654, -0.8106200654278459, 0.02513174339721196, 0.1731240139346307, 1.1742626244548813, -0.4140797033301196, 1.1727746600087567], [1.1647063631018333, -0.705427612754674, 1.5558996891703907, -0.625820143252845, -0.14961608608334884, -0.9118613593864469, 0.26005400314369137, 0.7922898875301918, 0.13186338400114206, -0.5850565537056434, 0.25922110545447763, -1.2376029852628825, 0.17486504393806274, -0.6635266515825161, 0.8269158303185137, -0.5017402423119174], [-1.0307767233886318, 1.0669037053945365, -0.1492489233392763, -0.17303413154887395, -1.1845807675556073, 0.7029843151791414, 0.44199508280162125, 0.5176339095885876, 0.5995911867511986, 0.8463496889238741, -0.19402432078757534, -0.7791464627609014, -0.2939157584675274, -0.8501426226800969, 0.8052813884887101, -0.2461695159869954], [0.7136441842577823, -0.6066907633802854, 0.26584245985876787, 0.12241457090843277, 0.8779707140973073, -1.1439636277875904, 0.10171577783190815, -1.2425893669958352, -0.48432650780890263, -0.2756907807922371, -3.7932028534577156, -0.7790177593022972, -1.7790560805589595, 0.2718331360849968, -0.8555219658901745, 0.5586224058940112]], [[-1.2602632632603512, 3.56579194873207, -2.3780448670258476, 1.3381955167352027, 0.060186026332028346, 0.594290930835549, -2.6296221747919852, -0.9152310701900064, 1.5658827378803366, -0.14074015083798874, 2.721594288920219, 0.4062908400928753, -1.7880344666226722, -0.18746054636297982, 1.170269953037726, 0.3348783620477456], [1.6250826484458498, 3.62500268453625, -2.3321191833935093, -0.1800789747022967, -0.8393825891829735, 0.4491193046039745, -2.7899877827982102, -0.2663859094673021, 0.41706261855418986, -1.0965997485726113, 1.3042593429157787, 0.7446587236735643, -0.4575801557276094, -0.6712219075729793, 0.872780791440975, -0.27362917665090575], [0.45525667973143347, 5.53994760151093, -2.545598832488442, -0.07736689164621499, -0.5952236312492881, 0.7825486001349768, -3.8185765970408716, 0.7316778737094043, 1.4842454793363755, -0.816268430005008, 1.9194725809402606, 0.3538889611009503, -1.1391211218185515, -1.400545579378526, 1.0651419940223512, -0.7208366194363957], [0.7417431891456103, 3.1308050600896165, -1.8025591154431053, 0.28350585652803323, -0.6000143031339499, 0.2621921273646019, -2.5953094151731535, 0.5357726428302392, 1.287882043973822, -0.5098020852521055, 2.051417342227579, 0.2218181316588463, -0.4064912458711621, -1.063404599082879, 1.8754091195081706, 0.35267349964839556], [0.03430787642058978, 3.207784039819293, -1.5166282113630445, -1.0670261007606454, -0.43648122585136695, -0.22315645380861127, -1.9831581321450293, 0.26731953133837594, 0.9187841234386104, -0.5858983604320555, 0.6111113111802374, 0.407871136948516, -1.2058371606344886, -0.36329001114841625, -0.05399310564009216, -0.3718103019278923], [0.14843983204791036, 2.007365871712885, -1.1756560987302855, 0.6887942980382369, -0.3337409236459567, 0.8558614086615794, -0.8813400969178967, 0.22249407755143671, 0.09490268443823213, -0.06290496533984402, -0.13932065275851846, -0.6802777459325983, -0.07999478955045218, -0.8370577881137709, 0.018966777036965237, -0.9911630204791512], [0.6780811551704672, 2.274003593719547, 0.9698467601606129, -0.145908324460848, 1.2322015825774149, -1.5619610472647467, -1.7951612598266815, 0.35061083843422547, -0.06736641444978925, -0.30113283214591624, 2.567052618755058, -0.16906874359833793, -1.9063127198451437, -0.29393323025422746, 0.7127819416866077, -1.1114389659267057]]], "dtype": "float64"}, {"__nd__": [[-0.20672769176909805, -0.02721899304154214, -0.5204117499243635, -0.42998912824774377, -0.036441722817208755, -0.08405835785808177, -0.1364136296536701, -0.14551302754108647], [-0.12149992155774694, -0.15965417239637653, -0.2530181278977545, 0.20270218616414348, 0.10307988746084147, 0.2890583851169318, 0.06891047675641684, -0.31122857218136857], [0.12652497138140395, 0.3858259567188917, 0.419345278006911, -0.05735907111392975, -0.18516883464155764, 0.14009546031725562, 0.3353635406393092, -0.12357502691138877], [-0.4669142640242115, -0.3424180036653678, 0.017606656483300842, 0.27152879878621966, 0.18727287187712494, -0.08461024886856586, -0.044448240734677895, 0.3487782250686797], [-0.013252004019690587, -0.35218326715695447, -0.1464006792895068, 0.35131964575431024, -0.26130921730236667, -0.3104465729494817, 0.2644511447529683, 0.24663118009206975], [-0.03822205964869941, -0.49833072439728077, -0.09745682624779337, 0.208919927531586, -0.11564489466400765, 0.30944774183180224, 0.04068187513614957, -0.004114911230581386], [-0.11843569302603131, -0.13907415036285803, 0.41028287253052065, -0.3199048662612456, 0.031284918807911025, 0.7614069078085683, -0.28216992383449624, -0.4924065136075577], [-0.08915235520058248, -0.14000446602120453, -0.3235861719713845, -0.06885222290260512, -0.37364159512026, 0.07088116403851161, -0.04076138626424433, -0.16167251822638196], [-0.3447580683514172, -0.07552392249775247, -0.13412459326562354, -0.1337146285085362, -0.026776855652296636, -0.09037559306968848, -0.3942897917187639, -0.060435765879123415], [0.014266517207742385, 0.016490963791731192, -0.08709530667583015, -0.23747922453590897, -0.13356066273333583, 0.09416954539935227, 0.6232011589294953, 0.21600986818578116], [-0.2962914624384713, 0.036969051162186906, 0.29399484728929426, 0.583742802478924, 0.15671753809312838, -0.03937998491890049, -0.13169366882094496, 0.07354005124862678], [0.43328516655455973, 0.0017761339165592305, 0.2599726238452007, -0.6031235987675958, -0.12745953962298906, -0.12643877186426752, -0.03270771034350378, 0.3549566994679318], [0.4269713300567673, -0.12809877103783474, 0.24888037873248453, 0.03166014016642923, -0.22772720681245376, 0.2573296762328975, -0.03329839815509694, -0.2953605678774957], [0.3802121603221206, -0.46668178748366895, -0.2146726157407663, -0.12245688456878137, 0.06443987610345193, -0.185550288643206, 0.12138204082814326, -0.09057613967864878], [0.0894050943039198, 0.19603706955682457, 0.018869390493185798, -0.029069231870649456, -0.34941209830777176, 0.14367484202776823, -0.1003807503917532, 0.4534968609477328], [0.4788773174736725, -0.2194978377678002, -0.027302924782716995, 0.12628222813764367, 0.04464676455349495, -0.21121810002108427, 0.2024338969632672, 0.10357345899922564]], "dtype": "float64"}, {"__nd__": [[0.13494719367273458, 0.08065407586882428, -0.41313069610927244, 0.30152972841700404, -0.4529232361665543, 0.17022257989964634, 0.2538485114725591, 0.04649230458728238], [0.0025473220954737204, -0.32766246741630556, 0.37178400800951344, -0.442200301426467, -0.2688815267506031, 0.18869530963550954, -0.14846418152270338, 0.5457982104406032], [0.21964401319196614, 0.33564891267151725, -0.03931476526563547, 0.4248573929459137, 0.09367693268623939, 0.36341219892989496, -0.04090273393395749, -0.00421938036066255], [0.06255228011564459, -0.06533610813786918, 0.2359779568086305, -0.20272679436748026, -0.2813741783735734, -0.0382005206089934, 0.030417590071270394, -0.37623695842476707], [0.07551849733002859, -0.21521631232192434, -0.16752987569254144, -0.11825556089413729, 0.13423092213828458, 0.059663872402650864, 0.38780948810929056, 0.13167533536631307], [-0.42730467456825083, -0.09789728007612664, -0.1321436893106524, 0.23298402173530167, 0.29651583255583785, 0.11424278643320701, 0.335119700992915, 0.0013010250080582597], [-0.006474731220415003, 0.3765678151206028, -0.3586931748840193, 0.0025379459347786015, 0.09435962365598012, -0.0037007858742897637, -0.49135867998890964, 0.27618412329364467], [-0.38876765547360015, 0.05164303970894313, 0.14491998677190066, 0.023317630291038616, -0.5193068986702318, 0.5149690237320056, 0.27339778434192286, 0.057954877270466484], [-0.19707233436501176, -0.4678444158873668, -0.16503864722070466, -0.47060532704850383, -0.8263444687400514, 0.1365037146423531, 0.11356746150156881, -0.1536391408629396], [-0.1719368862074011, 0.23166758478710717, -0.148546504937674, -0.05942140314138818, 0.21241194977900535, 0.16976624060759615, -0.2376752648421283, 0.42195130607374287], [0.37638205833888244, -0.5537324006126609, 0.0408325083852652, -0.0857056506563701, 0.29849778791153475, -0.3320236030810701, 0.5131579053919133, -0.21979597927007827], [0.1262011008006703, -0.1368408045156114, 0.2327225131184354, -0.6127379460930703, 0.27887320091319423, -0.05081632735088398, 0.1911940070055357, -0.002115480993816282], [0.22282185672218763, 0.2767277442787437, 0.25041880694932817, -0.07795771945097257, -0.308130345425892, 0.3938699757103302, -0.29218324864573036, 0.3602258947911111], [-0.304013853537234, -0.34106358792069313, 0.08033191904451582, -0.07019402185052727, 0.1670619132424288, -0.1746155854772588, 0.27224865471006826, 0.018016807762477965], [0.7447283937986668, -0.32554061006648344, -0.08183228830521805, 0.4619789944684406, -0.33064441827000385, 0.1908232126357279, -0.0828934953906571, -0.1407101774228492], [0.550084507348471, -0.10276761266489656, 0.15267254051950102, 0.18409630302254998, 0.1726718693307981, 0.1436374668173622, -0.24447193643274506, 0.1269034763739439]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(gqa_from_mha, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")